# Analisi Visuale degli Appalti Pubblici in Portogallo (PPP)
## Caso di Studio per l'Esame di Visualizzazione Dati e Visual Analytics (VDVAR)

**Studente:** Domenico Lacavalla
**Data:** 05/11/2025

---

### 1. Obiettivi e Contesto Teoretico

Questo notebook presenta un'analisi end-to-end del dataset "Public Procurement in Portugal" (PPP). L'obiettivo non è solo descrivere i dati, ma applicare i principi di **Information Visualization (InfoVis)** e **Visual Analytics (VA)** per estrarre conoscenza e "amplificare la cognizione", come definito da Shneiderman.

Il caso di studio segue l'intero processo di generazione di *insight*, dai dati grezzi alla conoscenza:
1.  **Dati (Raw Data):** Caricamento e ispezione.
2.  **Pre-processing & Analysis:** Applicazione del **Mantra di Visual Analytics di Keim** (*"Analyze first, Show the Important..."*). Eseguiamo prima un'analisi automatizzata (pulizia, feature engineering, clustering) per gestire la complessità e preparare i dati.
3.  **Visual Mapping:** Scelta deliberata delle tecniche di visualizzazione (il *Design Space*) in base al tipo di dati (temporali, n-D, gerarchici) e ai **canali visivi** più efficaci (es. posizione, lunghezza, colore).
4.  **Storytelling & Interazione:** Presentazione dei risultati seguendo il **Mantra della Ricerca di Informazioni di Shneiderman** (*"Overview first, zoom and filter, then details-on-demand"*).

### 2. Architettura del Notebook

L'architettura del codice è stata progettata per essere:
* **Modulare:** Separazione netta tra configurazione (`Config`), elaborazione (`DataCleaner`, `FeatureEngineer`) e visualizzazione (`BasePlotter`, `...Analyzer`).
* **Riproducibile:** Uso di percorsi relativi e configurazioni centralizzate per garantire che l'analisi sia verificabile.
* **Manutenibile:** Adozione del principio DRY (Don't Repeat Yourself) tramite classi base.

### 3. Setup dell'Ambiente e Configurazione

Per garantire la robustezza e la **riproducibilità** dell'analisi, tutte le costanti (percorsi file, nomi delle colonne, parametri grafici) sono centralizzate in una singola classe di configurazione (`Config`).

Questo approccio agisce da "Single Source of Truth", evitando valori *hardcoded* e facilitando la manutenzione. È un passo fondamentale di ingegneria del software che supporta l'affidabilità dell'intera pipeline di Visual Analytics.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from IPython.display import IFrame, display
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_white"
import warnings
warnings.filterwarnings("ignore")

class Config:
    """
    SINGLE SOURCE OF TRUTH.
    Centralizza tutte le configurazioni, percorsi e costanti del progetto.
    """
    # --- 1. Percorsi File (usando pathlib per compatibilità OS) ---
    BASE_DIR = Path.cwd()
    RAW_DATA = BASE_DIR / 'Datasets' / 'PPPData_EN_1.0.xlsx'
    CLEANED_DATA = BASE_DIR / 'Datasets' / 'PPPData_EN_cleaned_2.csv'
    GEOJSON = BASE_DIR / 'Datasets' / 'portugal_districts.geojson'
    PLOTS_DIR = BASE_DIR / 'plots_2'

    # --- 2. Nomi Colonne Chiave (per evitare typo nel codice) ---
    # Originali
    COL_ID = 'ID'
    COL_PRICE = 'Base Bid Price (€)'
    COL_DEADLINE = 'Execution deadline (days)'
    COL_DISTRICT = 'District'
    COL_YEAR = 'Signing Year'
    COL_DATE_SIGN = 'Signing date'
    COL_DATE_CLOSE = 'Closing date'
    COL_AWARD = 'Award criteria class'
    COL_CPVS = 'Cpvs Designation'
    
    # Generate/Derivate
    COL_PRICE_DAY = 'Price per Day'
    COL_DIFF_DATES = 'Days between close and signing'

    # Colonne da rimuovere perché ridondanti, vuote o non rilevanti per questa analisi
    DROP_COLS = [
        'Count', 'ID', 'Short Description1', 'Country', 'Award criteria',
        'Involves joint procurement (with several entities) (T/F)',
        'Awarded by a central purchasing body (T/F)',
        'Conclusion of a framework agreement (T/F)', 'Electronic auction (T/F)',
        'Negotiation phase (T/F)', 'Contracting by lots (T/F)', 'Collateral',
        'Contract end type', 'Justification for price change', 'Justification for deadline change'
    ]

    # Colonne che NON devono avere valori nulli per garantire un'analisi minima valida
    CRITICAL_COLS = [
        'Publication Year', 
        'Municipality', 
        'Base Bid Price (€)'
    ]
    
    # --- 3. Stile Visualizzazioni ---
    PALETTE = "viridis"
    COLOR_PRIMARY = "#3498DB"
    COLOR_SECONDARY = "#E74C3C"
    FIG_SIZE_STD = (12, 8)

    @staticmethod
    def setup():
        """Crea le directory necessarie se non esistono."""
        Config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Setup completato. Directory grafici: {Config.PLOTS_DIR}")

Config.setup()

### 4. Infrastruttura Software: `BasePlotter`

Per evitare la duplicazione del codice e garantire la coerenza visiva, è stata implementata una classe base `BasePlotter`.

**Motivazione Teorica (Design Space):**
Questa classe non è solo un'utilità di codice (principio DRY), ma è il nostro strumento per governare il **Design Space**. Centralizzando metodi come `_save` e `_format_currency`, imponiamo una "grammatica" visiva uniforme.

Essa standardizza:
1.  **Canali Visivi (Channels):** Assicura che la *palette* di colori e lo stile (dimensione font, sfondo) siano coerenti, riducendo il carico cognitivo dell'utente.
2.  **Substrato (Substrate):** Prepara lo spazio 2D per i grafici futuri.
3.  **Leggibilità:** Il metodo `_format_currency` (es. `€1M` invece di `1000000`) migliora la **leggibilità visiva**. Questo si collega allo studio "Beyond Memorability" (Borkin et al.), che evidenzia l'importanza della **ridondanza dei dati** (l'etichetta testuale rinforza il dato visivo) per migliorare la comprensione e il richiamo.

In [ ]:
class BasePlotter:
    """
    Classe genitore per tutte le visualizzazioni.
    Fornisce metodi di utilità condivisi per stile, formattazione e salvataggio.
    """
    def __init__(self):
        # Imposta il tema globale una volta per tutte
        sns.set_theme(style="whitegrid", context="talk", palette=Config.PALETTE)
        plt.rcParams['figure.figsize'] = Config.FIG_SIZE_STD
        plt.rcParams['axes.titleweight'] = 'bold'
        plt.rcParams['axes.titlesize'] = 16

    def _save(self, fig, filename: str):
        """Salva la figura in PNG (per report) gestendo il layout."""
        try: fig.tight_layout()
        except: pass
            
        path = Config.PLOTS_DIR / filename
        fig.savefig(path, dpi=150, bbox_inches='tight')
        print(f"Grafico salvato: {path.name}")
        plt.close(fig) # Chiude per liberare memoria

    def _format_currency(self, ax, axis='y', scale='M'):
        """
        Formatta gli assi numerici in formato valuta leggibile (€).
        scale: 'K' (migliaia), 'M' (milioni) o 'auto'.
        """
        def formatter(x, pos):
            if scale == 'auto':
                if x >= 1e9: return f'€{x*1e-9:.1f}B'
                if x >= 1e6: return f'€{x*1e-6:.1f}M'
                if x >= 1e3: return f'€{x*1e-3:.0f}K'
                return f'€{x:.0f}'
            elif scale == 'M': return f'€{x/1e6:.1f}M'
            elif scale == 'K': return f'€{x/1e3:.0f}K'
            return f'€{x:.0f}'

        func_fmt = FuncFormatter(formatter)
        if axis == 'y': ax.yaxis.set_major_formatter(func_fmt)
        else: ax.xaxis.set_major_formatter(func_fmt)

### 5. Fase 1: Data Loading & Ispezione Preliminare

Iniziamo il processo di Visual Analytics partendo dalla materia prima: i **Dati**. In questa fase, carichiamo il dataset grezzo e conduciamo una prima ispezione della sua integrità (valori nulli, tipi di dato).

**Scelta Tecnica:** Utilizziamo una classe dedicata `DataLoader` per incapsulare la logica di lettura. Questo rende l'analisi agnostica rispetto al formato del file sorgente (CSV, Excel) e migliora la modularità.

In [ ]:
class DataLoader:

    @staticmethod
    def load_raw() -> pd.DataFrame:
        path = Config.RAW_DATA
        print(f"Caricamento dati da: {path.name}...")
        try:
            if path.suffix in ['.xlsx', '.xls']:
                df = pd.read_excel(path)
            elif path.suffix == '.csv':
                df = pd.read_csv(path)
            else:
                raise ValueError("Formato non supportato")

            print(f"Dataset caricato: {df.shape[0]:,} righe, {df.shape[1]} colonne.")
            return df
        except Exception as e:
            print(f"Errore caricamento: {e}")
            return pd.DataFrame()

raw_df = DataLoader.load_raw()

print("\nInfo Dataset Grezzo:")
raw_df.info(memory_usage='deep')

### 5.1 Ispezione Visiva: Valori Mancanti

Prima di qualsiasi pulizia, è fondamentale *visualizzare* l'estensione dei dati mancanti. Le sole statistiche di riepilogo (come un `df.info()`) possono essere fuorvianti. Questo è il principio fondamentale dimostrato dall'**Anscombe's Quartet** e dal **Datasaurus Dozen**: set di dati con statistiche identiche possono avere strutture visive radicalmente diverse.

**Criteri di Scelta Tecnica (Bar Chart Orizzontale):**
Per questa analisi di integrità, un grafico a barre orizzontali è la scelta ottimale.
1.  **Canale Visivo Efficace:** Questa tecnica mappa un dato quantitativo (la percentuale di `null`) al canale visivo più efficace per il confronto: la **Posizione su un asse comune** (l'asse X) e la **Lunghezza**. Questa è in cima alla gerarchia dei canali visivi per i dati quantitativi.
2.  **Elaborazione Preattentiva:** L'uso della lunghezza, un potente **attributo preattentivo**, permette al nostro sistema percettivo di identificare *istantaneamente* le colonne più problematiche (le barre più lunghe) senza dover leggere ogni singolo valore.
3.  **Leggibilità:** L'orientamento orizzontale facilita la lettura delle etichette (nomi delle colonne) sull'asse Y.

In [ ]:
class IntegrityAnalyzer(BasePlotter):
    """Specializzata nell'analisi della qualità dei dati (es. missing values)."""
    
    def plot_missing_values(self, df: pd.DataFrame, title_suffix="") -> None:
        # Calcolo percentuali
        missing = df.isnull().mean() * 100
        missing = missing[missing > 0].sort_values(ascending=True)

        if missing.empty: print("Nessun valore mancante trovato!"); return

        # Creazione Plot
        fig, ax = plt.subplots(figsize=(10, max(6, len(missing) * 0.3)))
        
        # Usa la palette definita in Config
        bars = ax.barh(missing.index, missing.values, color=Config.COLOR_PRIMARY, alpha=0.8)
        
        # Styling
        ax.set_title('Analisi Integrità: Percentuale Valori Mancanti per Colonna', pad=20)
        ax.set_xlabel('Percentuale Mancante (%)')
        ax.set_xlim(0, 100)
        
        # Aggiunta etichette valore sulle barre
        for i, v in enumerate(missing.values): ax.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=10, color='#2C3E50')

        sns.despine()
        plt.show(fig)
        self._save(fig, f'01_missing_values_integrity_check_{title_suffix}.png')
        
integrity_checker = IntegrityAnalyzer()
integrity_checker.plot_missing_values(raw_df, "raw")

### 5.2 Definizione delle Colonne Critiche

Sulla base dell'analisi visiva precedente (che ha evidenziato colonne con molti *missing*) e della *domain knowledge* (comprensione del problema), definiamo formalmente nella `Config` le colonne da rimuovere (`DROP_COLS`) e quelle essenziali per l'analisi (`CRITICAL_COLS`).

Questo è un passo cruciale del **"Analyze first"** (Mantra di Keim): usiamo l'ispezione visiva per informare le nostre regole di pulizia automatizzata.

### 6. Data Cleaning (Mantra di Keim: "Analyze First")

Questa fase implementa la logica di pulizia e trasformazione. La classe `DataCleaner` incapsula tutte le operazioni.

**Contesto Teorico (Keim's VA Mantra):**
Con un dataset di grandi dimensioni, un "Overview" iniziale (come da mantra di Shneiderman) è spesso impossibile o inutile a causa del *noise*. Applichiamo quindi il mantra di Keim: **"Analyze first, Show the Important..."**.

La classe `DataCleaner` *è* il nostro "Analyze first": esegue un'analisi e una pulizia automatizzata (rimozione di colonne irrilevanti, gestione di tipi di dato eterogenei, rimozione di valori nulli critici) per produrre un sottoinsieme di dati pulito e affidabile, che sarà poi "mostrato" nelle fasi successive.

In [ ]:
class DataCleaner:
    """
    Gestisce la pulizia e la standardizzazione del DataFrame.
    Non effettua feature engineering, solo pulizia dei dati esistenti.
    """

    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df.copy()

    def clean_all(self) -> pd.DataFrame:
        """Esegue la pipeline completa di pulizia."""
        self._drop_redundant_columns()
        self._standardize_data_types()
        self._fix_specific_inconsistencies()
        self._remove_critical_missing()
        
        print(f"Pipeline di pulizia completata. Dimensioni finali: {self.df.shape}")
        return self.df

    def _drop_redundant_columns(self) -> None:
        """Rimuove le colonne definite in Config.DROP_COLS."""
        initial_cols = self.df.shape[1]
        self.df.drop(columns=[c for c in Config.DROP_COLS if c in self.df.columns], inplace=True)
        print(f"Colonne rimosse: {initial_cols - self.df.shape[1]}")

    def _standardize_data_types(self) -> None:
        """Normalizza i formati (es. booleani eterogenei, stringhe numeriche)."""
        # Standardizzazione Environmental criteria
        if 'Environmental criteria (T/F)' in self.df.columns:
            self.df['Environmental criteria (T/F)'] = (
                pd.to_numeric(self.df['Environmental criteria (T/F)'], errors='coerce')
                .fillna(0)
                .astype(int)
            )

        # Standardizzazione EU Journal publication
        if 'Published in the EU journal' in self.df.columns:
            mapping = {
                False: 0, 'False': 0, 0: 0, '0': 0,
                True: 1, 'True': 1, 'TRUE ': 1, 1: 1, '1': 1
            }
            self.df['Published in the EU journal'] = self.df['Published in the EU journal'].map(mapping).fillna(0).astype(int)

        # Pulizia stringhe Distretto
        if Config.COL_DISTRICT in self.df.columns: self.df[Config.COL_DISTRICT] = self.df[Config.COL_DISTRICT].astype(str).str.strip()

    def _fix_specific_inconsistencies(self) -> None:
        """Corregge errori noti specifici del dataset (business logic)."""
        # Rimozione incoerenze note nei codici distretto per Beja e Faro
        if 'District Code' in self.df.columns and Config.COL_DISTRICT in self.df.columns:
            mask_beja_error = (self.df[Config.COL_DISTRICT] == 'Beja') & (self.df['District Code'] == 13)
            mask_faro_error = (self.df[Config.COL_DISTRICT] == 'Faro') & (self.df['District Code'] == 13)
            
            rows_to_drop = self.df[mask_beja_error | mask_faro_error].index
            self.df.drop(rows_to_drop, inplace=True)
            self.df.drop(columns=['District Code'], inplace=True, errors='ignore')
            
            if len(rows_to_drop) > 0: print(f"Rimosse {len(rows_to_drop)} righe con incoerenze Distretto/Codice.")

    def _remove_critical_missing(self) -> None:
        """Rimuove righe che non hanno dati sufficienti per l'analisi base."""
        initial_rows = len(self.df)
        # Verifica quali colonne critiche esistono effettivamente nel df
        existing_critical = [col for col in Config.CRITICAL_COLS if col in self.df.columns]
        self.df.dropna(subset=existing_critical, inplace=True)
        dropped = initial_rows - len(self.df)
        if dropped > 0: print(f"Rimosse {dropped} righe con valori mancanti in campi critici {existing_critical}.")

# --- ESECUZIONE CLEANING ---
cleaner = DataCleaner(raw_df)
cleaned_df = cleaner.clean_all()

### 6.1 Verifica Integrità Post-Pulizia

L'analisi visiva non è un processo lineare, ma un *ciclo iterativo*. Dopo aver eseguito la pulizia automatizzata, rieseguiamo l'analisi visiva dei valori mancanti.

Questo *feedback loop* è fondamentale per:
1.  **Validare:** Confermare che le operazioni di pulizia (es. `_remove_critical_missing`) abbiano avuto successo.
2.  **Scoprire:** Identificare eventuali problemi residui (es. colonne non critiche ancora con molti *missing*) che potrebbero richiedere un'imputazione o un'analisi più approfondita.

In [ ]:
integrity_checker.plot_missing_values(cleaned_df, "cleaned")

### 7. Feature Engineering

Il Feature Engineering è il processo di trasformazione dei dati grezzi in *feature* che ne aumentano il potenziale esplicativo. È un passo fondamentale che precede il **Visual Mapping**.

Dati grezzi (come stringhe di date o descrizioni testuali) sono spesso **dati astratti** che, secondo la distinzione **InfoVis vs. SciVis**, non hanno una mappatura geometrica o fisica ovvia. Il nostro compito è trasformarli in dimensioni analizzabili:
1.  **Dati Temporali:** Convertiamo stringhe (es. '05/11/2025') in oggetti `datetime` e ne estraiamo componenti (Anno, Mese). Calcoliamo anche metriche derivate (es. `Days between close and signing`) che possono rivelare *pattern* o inefficienze.
2.  **Dati Finanziari (n-D):** Creiamo metriche normalizzate (es. 'Price per Day'). Questo è essenziale per confrontare contratti con durate diverse, altrimenti un confronto basato solo sul prezzo totale sarebbe fuorviante.
3.  **Dati Testuali (1D-Lineari):** Come vedremo, trasformeremo il testo libero (dati 1D) in *feature* strutturate usando l'NLP.

In [ ]:
class FeatureEngineer:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def engineer_all(self) -> pd.DataFrame:
        self._engineer_dates()
        self._engineer_financials()
        print(f"Feature Engineering completato. Nuove dimensioni: {self.df.shape}")
        return self.df

    def _engineer_dates(self):
        date_cols = [Config.COL_DATE_SIGN, Config.COL_DATE_CLOSE]
        # Aggiungiamo 'Publication date' se esiste, anche se non è in Config
        if 'Publication date' in self.df.columns:
             date_cols.append('Publication date')

        for col in date_cols:
            if col not in self.df.columns: continue
            self.df[col] = pd.to_datetime(self.df[col], errors='coerce', dayfirst=True, infer_datetime_format=True)
            
            # Estrae anno e mese
            year_col = f"{col.split()[0]} Year"
            month_col = f"{col.split()[0]} Month"
            self.df[year_col] = self.df[col].dt.year
            self.df[month_col] = self.df[col].dt.month

        # Calcolo differenza giorni
        if all(c in self.df.columns for c in [Config.COL_DATE_CLOSE, Config.COL_DATE_SIGN]):
            self.df[Config.COL_DIFF_DATES] = (self.df[Config.COL_DATE_CLOSE] - self.df[Config.COL_DATE_SIGN]).dt.days

    def _engineer_financials(self):
        if Config.COL_PRICE in self.df.columns and Config.COL_DEADLINE in self.df.columns:
            safe_deadline = self.df[Config.COL_DEADLINE].replace(0, np.nan)
            self.df[Config.COL_PRICE_DAY] = self.df[Config.COL_PRICE] / safe_deadline
            self.df[Config.COL_PRICE_DAY].replace([np.inf, -np.inf], np.nan, inplace=True)

engineer = FeatureEngineer(cleaned_df)
processed_df = engineer.engineer_all()

### 7.1 Feature Engineering Testuale (NLP)

Le descrizioni dei contratti (CPV) sono **dati 1D-Lineari** non strutturati. Per analizzarli visivamente, dobbiamo prima trasformarli in dati **multidimensionali (n-D)**.

**Perché TF-IDF?**
Utilizziamo **TF-IDF (Term Frequency-Inverse Document Frequency)**. A differenza di un semplice conteggio, TF-IDF penalizza le parole troppo comuni (che appaiono in tutti i contratti e quindi non discriminano) ed esalta quelle specifiche di pochi contratti.

Questo è un altro esempio di **"Analyze First"**: usiamo un modello algoritmico per estrarre *feature* latenti (le *keyword* tematiche) dal testo, che verranno poi usate come dimensioni per l'analisi visiva (es. in grafici a barre o mappe semantiche). Questo approccio è concettualmente simile a quello di **VisRA (Visual Readability Analysis)**, che analizza feature linguistiche (come la complessità del vocabolario) per supportare l'analisi.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
import re
import string

try:
    nlp_engine = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
except OSError:
    print("Modello spaCy 'en_core_web_sm' non trovato. Esegui: python -m spacy download en_core_web_sm")
    nlp_engine = None

class TextFeatureEngineer(FeatureEngineer):
    """Estende FeatureEngineer con capacità specifiche di NLP."""
    
    def engineer_text(self, text_col: str, num_keywords: int = 10) -> pd.DataFrame:
        """
        Estrae keyword principali da una colonna testuale usando TF-IDF.
        Crea colonne booleane per la presenza di ciascuna keyword top.
        """
        if nlp_engine is None or text_col not in self.df.columns: return self.df

        print(f"Inizio elaborazione testuale su '{text_col}'...")
        clean_text = self.df[text_col].astype(str).apply(self._preprocess_text)
        self.df[f'{text_col}_cleaned'] = clean_text # Salva testo pulito per usi futuri (es. WordCloud)

        # 2. TF-IDF per estrazione keyword
        tfidf = TfidfVectorizer(max_features=num_keywords, ngram_range=(1, 2), stop_words='english')
        try:
            tfidf_matrix = tfidf.fit_transform(clean_text)
            keywords = tfidf.get_feature_names_out()
            print(f" Top {num_keywords} keyword estratte: {list(keywords)}")

            # 3. Creazione colonne booleane per keyword
            for keyword in keywords:
                safe_col_name = f"cpvs_keyword_{re.sub(r'[^a-zA-Z0-9]', '_', keyword)}"
                self.df[safe_col_name] = clean_text.str.contains(keyword, regex=False).astype(int)
                
        except ValueError as e:
            print(f"Errore TF-IDF (possibile testo insufficiente): {e}")

        return self.df

    @staticmethod
    def _preprocess_text(text: str) -> str:
        """Pulisce una singola stringa (lowercase, no punctuation, lemmatization)."""
        text = text.lower().translate(str.maketrans('', '', string.punctuation + string.digits))
        doc = nlp_engine(text)
        return " ".join([t.lemma_ for t in doc if not t.is_stop and len(t.lemma_) > 2])

# --- ESECUZIONE FEATURE ENGINEERING TESTUALE ---
text_engineer = TextFeatureEngineer(processed_df)
final_df = text_engineer.engineer_text(Config.COL_CPVS, num_keywords=10)

### 8. Analisi e Gestione Distribuzioni (Visual Outlier Detection)

Prima di procedere, è essenziale comprendere la distribuzione delle variabili numeriche chiave. Le statistiche di riepilogo (come media, min, max) sono notoriamente ingannevoli e possono nascondere la vera natura dei dati, come dimostrato dall'**Anscombe's Quartet** e dal **Datasaurus Dozen**.

**Criteri di Scelta Tecnica (Istogramma + Box Plot):**
Per evitare queste trappole, usiamo una combinazione di visualizzazioni:
1.  **Istogramma:** Ci permette di vedere la *forma* della distribuzione (es. skewness, multimodalità).
2.  **Box Plot:** Utilizza il canale della **Posizione** per mostrare robusti indicatori statistici (mediana, quartili) e identificare formalmente gli *outlier* (i punti oltre i baffi).

Questa combinazione ci fornisce una visione completa e robusta, permettendoci di identificare visivamente i valori estremi che potrebbero distorcere le analisi successive.

In [ ]:
class DistributionAnalyzer(BasePlotter):
    """Visualizza le distribuzioni per identificare outlier."""
    
    def plot_distribution(self, df: pd.DataFrame, cols: list[str]):
        for col in cols:
            if col not in df.columns: continue
            
            # Setup figura doppia (Istogramma + Boxplot)
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            fig.suptitle(f"Distribuzione di '{col}' (Pre-pulizia)", fontweight='bold')
            
            # 1. Istogramma con stima densità (KDE)
            sns.histplot(df[col].dropna(), kde=True, ax=axes[0], 
                         color=Config.COLOR_PRIMARY, alpha=0.6, edgecolor='black')
            axes[0].set_title('Istogramma e Densità')
            
            # 2. Box Plot (evidenzia outlier come punti)
            sns.boxplot(x=df[col].dropna(), ax=axes[1], 
                        color=Config.COLOR_SECONDARY, width=0.5, flierprops={'markerfacecolor':'red'})
            axes[1].set_title('Box Plot (Outlier in rosso)')
            
            sns.despine()
            plt.show()
            self._save(fig, f'02a_distribution_{col.replace(" ", "_").lower()}.png')         

# --- ESECUZIONE VISUALIZZAZIONE ---
dist_analyzer = DistributionAnalyzer()
cols_to_check = [Config.COL_DEADLINE, Config.COL_DIFF_DATES]
if Config.COL_PRICE in final_df.columns: cols_to_check.append(Config.COL_PRICE)
dist_analyzer.plot_distribution(final_df, cols_to_check)

### 8.1 Rimozione Outlier e Imputazione Finale

L'analisi visiva della cella precedente (Fase 8) ha confermato la presenza di outlier estremi. Ora agiamo su questa scoperta, seguendo il ciclo di Visual Analytics: **Analyze -> Show -> Analyze Further**.

1.  **Rimozione Outlier (Filter):** Applichiamo un filtro statistico (basato sull'Interquartile Range, IQR) per rimuovere i valori estremi che non rappresentano il "comportamento tipico" dei dati e che distorcerebbero le medie e le scale dei grafici futuri. Questo è un task di **Filtro** nel senso del mantra di Shneiderman.
2.  **Imputazione Finale:** Gestiamo i valori mancanti residui (es. in `COL_DEADLINE`) usando la *mediana* (più robusta della media agli outlier che potrebbero essere ancora presenti).

Questo ci assicura un dataset finale completo e stabile, pronto per l'analisi visiva finale.

In [ ]:
class DataRefiner:
    """Gestisce imputazione valori mancanti e rimozione outlier."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def refine_all(self) -> pd.DataFrame:
        self._remove_outliers([Config.COL_DEADLINE, Config.COL_DIFF_DATES])
        self._impute_missing()
        return self.df

    def _remove_outliers(self, cols: list[str]):
        """Rimuove righe esterne a 2.5*IQR per le colonne specificate."""
        for col in cols:
            if col not in self.df.columns: continue
            
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 2.5 * IQR
            upper = Q3 + 2.5 * IQR
            
            initial_rows = len(self.df)
            # Manteniamo i NaN qui (saranno gestiti dall'imputer se necessario), filtriamo solo i valori validi ma estremi
            mask = (self.df[col].isna()) | ((self.df[col] >= lower) & (self.df[col] <= upper))
            self.df = self.df[mask]
            
            removed = initial_rows - len(self.df)
            if removed > 0: print(f"Rimossi {removed} outlier da '{col}' (Range accettato: [{lower:.1f}, {upper:.1f}])")

    def _impute_missing(self):
        """Riempie i valori mancanti residui con strategie standard."""
        # Esempio: riempie la scadenza mancante con la mediana (più robusta della media)
        if Config.COL_DEADLINE in self.df.columns and self.df[Config.COL_DEADLINE].isna().any():
             median_val = self.df[Config.COL_DEADLINE].median()
             self.df[Config.COL_DEADLINE].fillna(median_val, inplace=True)
             print(f"Imputati valori mancanti in '{Config.COL_DEADLINE}' con la mediana: {median_val:.0f}")

# --- ESECUZIONE REFINING ---
refiner = DataRefiner(final_df)
refined_df = refiner.refine_all()

### 8.2 Feature Engineering: Discretizzazione

L'ultima fase di preparazione dei dati è la *discretizzazione*. Trasformiamo variabili numeriche continue (come `COL_PRICE`) in variabili categoriche (es. 'Low', 'Medium', 'High').

**Motivazione Teorica (Design Space):**
Questa trasformazione è potente perché "sblocca" l'uso di canali visivi diversi e molto efficaci:
* Una variabile **Quantitativa** (Price) è mappata efficacemente a *Lunghezza* o *Posizione*.
* Trasformandola in **Nominale/Categorica**, possiamo ora mapparla a **Tonalità del Colore (Color Hue)** o usarla per raggruppare (es. in *small multiples* o *stacked bars*).

Questo arricchisce il nostro "Design Space", permettendoci di segmentare le visualizzazioni e rispondere a domande come: "Come si distribuiscono i contratti 'High' (di alto valore) tra i vari distretti?"

In [ ]:
def add_discrete_features(df: pd.DataFrame) -> pd.DataFrame:
    """Aggiunge versioni categoriche delle feature numeriche principali."""
    df = df.copy()
    
    # 1. Discretizzazione Scadenze
    for col in [Config.COL_DEADLINE, Config.COL_DIFF_DATES]:
        if col in df.columns:
            new_col = f"{col}_cat"
            try: df[new_col] = pd.qcut(df[col], 3, labels=['Short', 'Medium', 'Long'])
            except ValueError: df[new_col] = pd.cut(df[col], 3, labels=['Short', 'Medium', 'Long'])

    # 2. Discretizzazione Prezzo
    if Config.COL_PRICE in df.columns:
        new_col = f"{Config.COL_PRICE}_cat"
        try: df[new_col] = pd.qcut(df[Config.COL_PRICE], 3, labels=['Low', 'Medium', 'High'])
        except: median = df[Config.COL_PRICE].median(); df[new_col] = pd.cut(df[Config.COL_PRICE], bins=[-np.inf, median, np.inf], labels=['Low', 'High'])
            
    print("Feature discrete aggiunte (suffisso '_cat').")
    return df

refined_df = add_discrete_features(refined_df)

### 9. Analisi Testuale Avanzata (Clustering Semantico)

Portiamo l'approccio **"Analyze First"** al livello successivo. Invece di guardare solo le *keyword* (TF-IDF), vogliamo capire i "temi" semantici latenti nei testi dei contratti.

**Pipeline Analitica:**
1.  **Embedding:** Usiamo un modello (Sentence Transformer) per convertire ogni descrizione testuale (dato 1D) in un vettore numerico ad alta dimensionalità (dato n-D) che cattura il *significato semantico*.
2.  **Clustering:** Applichiamo K-Means a questi vettori per raggruppare i contratti in *cluster* tematici (es. "Lavori stradali", "Servizi IT", "Costruzione edifici").

**Valore Analitico:**
Questo processo crea una nuova, potentissima variabile **Nominale** (il `semantic_cluster`). Questo è un puro esempio di **InfoVis**: abbiamo preso dati astratti (testo) e, tramite l'analisi, abbiamo *creato* una struttura che ora possiamo visualizzare e utilizzare per segmentare l'analisi finanziaria (es. "Quanto valgono mediamente i contratti del cluster 'Lavori stradali'?").

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer


class SemanticClusterer:
    """Clustering semantico con analisi finanziaria integrata."""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def run_clustering(self, text_col: str, n_clusters=5) -> pd.DataFrame:
        if text_col not in self.df.columns: return self.df

        print(f"Avvio Clustering Semantico su '{text_col}'...")
        try:
            model = SentenceTransformer('all-MiniLM-L6-v2')
            embeddings = model.encode(self.df[text_col].fillna("").astype(str).tolist(), 
                                    show_progress_bar=True, batch_size=128)        
            pca = PCA(n_components=2, random_state=42)
            coords = pca.fit_transform(embeddings)
            self.df['semantic_x'] = coords[:, 0]
            self.df['semantic_y'] = coords[:, 1]
            
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            self.df['semantic_cluster'] = kmeans.fit_predict(embeddings)
            print(f"Clustering completato: {n_clusters} gruppi individuati.")
            
            self._analyze_financials('semantic_cluster')
            
        except Exception as e:
            print(f"Errore nel clustering (librerie mancanti?): {e}")
            
        return self.df

    def _analyze_financials(self, cluster_col: str):
        """Stampa il profilo finanziario di ogni cluster."""
        if Config.COL_PRICE not in self.df.columns: return
        
        print("\nProfilo Finanziario per Cluster Semantico:")
        stats = self.df.groupby(cluster_col)[Config.COL_PRICE].agg(
            N=('count'),
            Valore_Medio=('mean'),
            Valore_Mediano=('median'),
            Totale=('sum')
        ).sort_values(by='Valore_Medio', ascending=False)
        
        # Formattazione per output leggibile
        for col in ['Valore_Medio', 'Valore_Mediano', 'Totale']: stats[col] = stats[col].map('€{:,.0f}'.format)       
        print(stats)

# --- ESECUZIONE CLUSTERING ---
clusterer = SemanticClusterer(refined_df)
col_to_cluster = Config.COL_CPVS
clustered_df = clusterer.run_clustering(col_to_cluster, n_clusters=5)

### 10. Sommario Finale e Checkpoint di Preprocessing

Prima di passare alla Fase 2 (Visualizzazione), stampiamo un "certificato di buona salute" del dataset.

Questo riepilogo testuale funge da *checkpoint* finale, confermando le dimensioni, i nuovi tipi di dato e le statistiche di base. Ci assicura che il risultato della nostra complessa pipeline **"Analyze First"** sia un dataset robusto, arricchito e pronto per l'esplorazione visiva, che ci permetterà di generare *insight*.

In [ ]:
class DataSummarizer:
    """Genera un report testuale riassuntivo del dataset pronto."""
    
    @staticmethod
    def print_summary(df: pd.DataFrame):
        print("\n" + "="*40)
        print("DATASET MASTER: RIEPILOGO FINALE")
        print("="*40)
        print(f"Dimensioni: {df.shape[0]:,} righe, {df.shape[1]} colonne")
        print("\n--- Tipi di Dato ---")
        print(df.dtypes.value_counts())
        
        print("\n--- Statistiche Chiave (Numeriche) ---")
        # Seleziona solo alcune colonne chiave per non intasare l'output
        cols_to_summarize = [Config.COL_PRICE, Config.COL_DEADLINE, Config.COL_DIFF_DATES]
        cols_existing = [c for c in cols_to_summarize if c in df.columns]
        if cols_existing:
            print(df[cols_existing].describe().T[['mean', '50%', 'min', 'max']])

        print("\n--- Anteprima Cluster (se presenti) ---")
        if 'semantic_cluster' in df.columns:
            print(df['semantic_cluster'].value_counts().sort_index())
        
        print("="*40 + "\n")

# --- ESECUZIONE SOMMARIO E SALVATAGGIO FINALE ---
DataSummarizer.print_summary(clustered_df)

### 11. Salva Checkpoint

Salviamo il dataset "master" processato. Questo checkpoint fisico (`CLEANED_DATA`) è fondamentale per la **riproducibilità** e l'efficienza. Ci permette di ricaricare i dati puliti e arricchiti per la Fase 2 senza dover rieseguire l'intera pipeline di preprocessing (che, specialmente con l'NLP e il clustering, può essere computazionalmente costosa).

In [ ]:
clustered_df.to_csv(Config.CLEANED_DATA, index=False)
print(f"Dataset MASTER salvato in: {Config.CLEANED_DATA}")
print("Pronto per la Fase 2: Visualizzazione.")

In [ ]:
if Path(Config.CLEANED_DATA).exists():
    df_master = pd.read_csv(Config.CLEANED_DATA)
    for col in ['Signing date', 'Closing date']:
        if col in df_master.columns:
             df_master[col] = pd.to_datetime(df_master[col])
    print(f"Dataset Master caricato per la visualizzazione: {df_master.shape}")
else:
    print("ATTENZIONE: File dati puliti non trovato. Eseguire prima la Fase 1.")
    df_master = clustered_df

# FASE 2: VISUALIZATION & STORYTELLING

Con un dataset robusto e arricchito, entriamo nella fase di esplorazione visiva. L'obiettivo è applicare i principi teorici di InfoVis per "amplificare la cognizione" e generare *insight*.

## Principio Organizzativo: Mantra di Shneiderman e Keim

L'intera analisi visuale è strutturata attorno ai due mantra fondamentali della Visual Analytics:

1.  **Mantra del Visual Analytics (Keim):** Avendo già completato il passo **"Analyze first"** nella Fase 1, ora ci concentriamo su **"Show the Important"** (mostrare i pattern aggregati e i cluster), **"Zoom, filter and analyze further"** (permettere l'approfondimento) e **"Details on demand"**.

2.  **Mantra della Ricerca di Informazioni (Shneiderman):** Lo storytelling di questa fase seguirà la progressione classica:
    * **Overview First:** Inizieremo con una dashboard KPI (`DashboardBuilder`) per fornire una visione d'insieme dell'intero fenomeno.
    * **Zoom and Filter:** Utilizzeremo analizzatori specializzati (`TemporalAnalyzer`, `GeospatialAnalyzer`) per approfondire dimensioni specifiche (tempo, geografia) e filtrare i dati.
    * **Details on Demand:** Impegheremo grafici interattivi (Plotly) e tabelle che, tramite *hover* (il task *Details-on-demand*) e *lookup*, forniscono dettagli puntuali senza sovraccaricare la vista principale.

## Architettura Software

L'implementazione segue il principio di responsabilità singola. Ogni classe `Analyzer` si concentra su un tipo di dati o un task analitico, ereditando da `BasePlotter` per garantire coerenza nel **Design Space**.

## Note Metodologiche Visuali

* **Scala Logaritmica:** Utilizzata estensivamente per i dati finanziari (prezzo). Come visto nella Fase 1, questi dati sono *right-skewed*. La scala logaritmica è una trasformazione che ci permette di visualizzare e confrontare ordini di grandezza diversi (da €1k a €100M) nello stesso grafico, senza che i valori più piccoli vengano schiacciati a zero.
* **Mappe Coropletiche:** Tecnica classica per **Dati 2D-Map**. Si tratta di un substrato geografico (che ricade nella **SciVis**) su cui mappiamo dati astratti (un task di **InfoVis**), come il prezzo medio o il volume, usando il canale del **Colore (Saturazione)**.
* **KDE (Kernel Density Estimation):** Usato per l'analisi di densità bivariata. È superiore a uno scatterplot standard in presenza di *overplotting* (migliaia di punti sovrapposti), poiché stima e visualizza la densità (spesso tramite *colore*) permettendo di identificare i *cluster*.

## 1. Panoramica Esecutiva (KPI Dashboard)

Iniziamo la nostra analisi seguendo la prima regola del mantra di Shneiderman: **"Overview first"**.

Questa dashboard composita funge da "cruscotto" e fornisce il contesto generale. Combina indicatori chiave di performance (KPI) numerici (per un *lookup* immediato dei valori) con grafici di alto livello (per la percezione dei *pattern*). L'obiettivo è dare all'utente un orientamento immediato sulle dimensioni del fenomeno:
* **Volume e Valore Totale:** Quanto è grande il dataset?
* **Durata Media:** Quanto tempo richiedono i progetti?
* **Top Player:** Quali distretti muovono più denaro?
* **Trend Generale:** Il mercato è in crescita o contrazione?

In [ ]:
import matplotlib.gridspec as gridspec

class DashboardBuilder(BasePlotter):
    """Costruisce dashboard riepilogative complesse."""
    
    def build_main_kpi(self, df: pd.DataFrame):
        fig = plt.figure(figsize=(18, 12))
        gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)
        fig.patch.set_facecolor('#F8F9FA') # Sfondo leggero professionale

        # --- RIGA 1: KPI CARDS ---
        kpi1 = fig.add_subplot(gs[0, 0])
        self._draw_kpi_card(kpi1, f"{len(df):,}", "Contratti Totali", "Dataset analizzato", Config.COLOR_PRIMARY)
        
        kpi2 = fig.add_subplot(gs[0, 1])
        avg_val = df[Config.COL_PRICE].mean()
        self._draw_kpi_card(kpi2, f"€{avg_val/1e6:.1f}M", "Valore Medio", "Per contratto", "#2ECC71")
        
        kpi3 = fig.add_subplot(gs[0, 2])
        avg_days = df[Config.COL_DEADLINE].mean()
        self._draw_kpi_card(kpi3, f"{avg_days:.0f}", "Giorni Medi", "Durata esecuzione", Config.COLOR_SECONDARY)

        # --- RIGA 2: ANALISI DISTRETTI & PREZZI ---
        # Top 5 Distretti per Valore
        ax_dist = fig.add_subplot(gs[1, :2])
        top_5 = df.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].sum().nlargest(5).sort_values(ascending=True)
        bars = ax_dist.barh(top_5.index, top_5.values, color=sns.color_palette("viridis", 5))
        ax_dist.set_title("Top 5 Distretti per Valore Totale Contratti", fontweight='bold')
        self._format_currency(ax_dist, 'x', 'M')
        sns.despine(ax=ax_dist, left=True)

        # Distribuzione Prezzi (Boxen Plot per gestire code lunghe)
        ax_price = fig.add_subplot(gs[1, 2])
        if Config.COL_AWARD in df.columns:
            sns.boxenplot(data=df, x=Config.COL_AWARD, y=Config.COL_PRICE, ax=ax_price, palette="Set2")
            ax_price.set_yscale('log')
            ax_price.set_title("Distribuzione Prezzi per Criterio", fontweight='bold')
            ax_price.set_ylabel("Prezzo (€, scala log)")
            ax_price.set_xlabel("")
            plt.setp(ax_price.get_xticklabels(), rotation=15, ha="right")

        # --- RIGA 3: TREND TEMPORALE (Doppio Asse) ---
        ax_trend = fig.add_subplot(gs[2, :])
        yearly = df.groupby(Config.COL_YEAR).agg(
            Count=(Config.COL_YEAR, 'size'), 
            Value=(Config.COL_PRICE, 'sum')
        )
        
        # Linea Volume (sx)
        ax_trend.plot(yearly.index, yearly['Count'], marker='o', color=Config.COLOR_PRIMARY, lw=3, label='N. Contratti')
        ax_trend.set_ylabel('Numero Contratti', color=Config.COLOR_PRIMARY, fontweight='bold')
        ax_trend.tick_params(axis='y', labelcolor=Config.COLOR_PRIMARY)
        
        # Barre Valore (dx)
        ax_val = ax_trend.twinx()
        ax_val.bar(yearly.index, yearly['Value'], color=Config.COLOR_SECONDARY, alpha=0.5, label='Valore Totale')
        ax_val.set_ylabel('Valore Totale (€)', color=Config.COLOR_SECONDARY, fontweight='bold')
        ax_val.tick_params(axis='y', labelcolor=Config.COLOR_SECONDARY)
        self._format_currency(ax_val, 'y', 'M')
        
        ax_trend.set_title("Trend Temporale: Volume vs Valore", fontweight='bold')
        
        # Legenda unica manuale
        lines, labels = ax_trend.get_legend_handles_labels()
        lines2, labels2 = ax_val.get_legend_handles_labels()
        ax_trend.legend(lines + lines2, labels + labels2, loc='upper left')

        plt.suptitle('Dashboard Analitica - Appalti Pubblici Portogallo', fontsize=22, fontweight='bold', y=0.95)
        plt.show()
        self._save(fig, '10_kpi_dashboard_executive.png')

    def _draw_kpi_card(self, ax, value, title, subtitle, color):
        """Helper interno per disegnare una 'card' KPI pulita."""
        ax.axis('off')
        # Rettangolo di sfondo con bordo colorato
        rect = plt.Rectangle((0.05, 0.05), 0.9, 0.9, transform=ax.transAxes, 
                             fc='white', ec=color, lw=2, alpha=1, zorder=1)
        ax.add_patch(rect)
        
        ax.text(0.5, 0.6, value, transform=ax.transAxes, ha='center', va='center', 
                fontsize=32, fontweight='bold', color=color, zorder=2)
        ax.text(0.5, 0.35, title, transform=ax.transAxes, ha='center', va='center', 
                fontsize=14, fontweight='bold', color='#2C3E50', zorder=2)
        ax.text(0.5, 0.2, subtitle, transform=ax.transAxes, ha='center', va='center', 
                fontsize=10, color='#95A5A6', zorder=2)

# --- ESECUZIONE DASHBOARD ---
dashboard = DashboardBuilder()

### 1.1 Interpretazione Dashboard KPI

#### Overview First - Panoramica Dataset

**Elementi Visualizzati:**
* **KPI Cards**: Sintesi numerica. Come evidenziato nello studio "Beyond Memorability" (Borkin et al.), il **testo e i titoli sono fondamentali** e la **ridondanza dei dati** (mostrare un numero chiave in grande) aiuta la memorizzazione e la comunicazione.
* **Top 5 Distretti (Bar Chart)**: Utilizza il canale visivo più efficace per i confronti quantitativi: **Lunghezza** su un asse comune.
* **Distribuzione Prezzi per Criterio (Boxen Plot)**: Un Boxen plot (un'evoluzione del box plot) è scelto per visualizzare meglio la forma della distribuzione di un gran numero di punti. Mappa le statistiche (quantili) alla **Posizione** sull'asse Y.
* **Trend Volume vs Valore (Dual-Axis)**: Questo grafico mappa due variabili quantitative (Conteggio e Valore) sullo stesso asse temporale (X) ma su due assi Y distinti (con scale diverse). È una tecnica potente ma complessa, usata qui per evidenziare la *correlazione* tra i due trend.

**Criteri di Scelta Tecnica:**
* **GridSpec Layout**: Organizza le visualizzazioni in una gerarchia visiva chiara.
* **Boxen Plot**: Superiore al box plot standard per *big data*, in quanto mostra più quantili e dà una stima migliore della forma della distribuzione.
* **Dual-Axis**: Consente il confronto diretto di trend con scale incomparabili (numero di contratti vs. miliardi di euro).

**Analisi Dati Osservati:**
* **Concentrazione Geografica**: I 5 distretti principali assorbono la maggior parte del valore.
* **Segmentazione per Criterio**: Contratti aggiudicati con "Qualità/Prezzo" (MEAT) presentano mediane superiori rispetto a "Prezzo Più Basso".
* **Elasticità Temporale**: Forte correlazione positiva tra volume (N. contratti) e valore (€), con picchi e valli coincidenti.

In [ ]:
dashboard.build_main_kpi(df_master)

## 2. Analisi Temporale e Stagionalità

Dopo l'Overview, passiamo alla fase di **"Zoom and Filter"**. In questa sezione, "zoomiamo" sulla dimensione temporale per analizzare **dati temporali (Time-Series)**. L'obiettivo è scoprire pattern ciclici (stagionalità) e trend evolutivi.

In [ ]:
class TemporalAnalyzer(BasePlotter):
    """Analisi di trend, stagionalità ed evoluzione temporale."""
    
    def plot_seasonality_heatmap(self, df: pd.DataFrame):
        """Genera heatmap stagionale Mese × Anno."""
        if 'Signing Month' not in df.columns: 
            print("Colonna 'Signing Month' non trovata")
            return
        
        heatmap_data = df.pivot_table(
            index='Signing Month', columns=Config.COL_YEAR, 
            values=Config.COL_ID if Config.COL_ID in df.columns else df.columns[0], 
            aggfunc='count'
        ).fillna(0)
        
        fig, ax = plt.subplots(figsize=(14, 8))
        # Heatmap con palette professionale blu-bianco-rosso
        sns.heatmap(heatmap_data, cmap='RdYlBu_r', annot=True, fmt='.0f', 
                   linewidths=1, linecolor='white', ax=ax, 
                   cbar_kws={
                       'label': 'N. Contratti',
                       'shrink': 0.8,
                       'pad': 0.02,
                       'aspect': 30
                   }, 
                   vmin=0, annot_kws={'fontsize': 10, 'weight': 'bold'})
        
        ax.set_title("Heatmap Stagionale: Intensità Contratti per Mese/Anno", 
                    fontweight='bold', fontsize=16, pad=20, color='#2C3E50')
        ax.set_xlabel("Anno", fontweight='bold', fontsize=12)
        ax.set_ylabel("Mese", fontweight='bold', fontsize=12)
        ax.set_yticklabels(['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 
                            'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic'], rotation=0)
        
        # Colorbar a destra più visibile
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=11)
        cbar.set_label('N. Contratti', fontsize=12, weight='bold')
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '11_seasonality_heatmap.png')
    
    def plot_criteria_evolution(self, df: pd.DataFrame):
        """Genera stacked area chart evoluzione criteri."""
        if Config.COL_AWARD not in df.columns: 
            print("Colonna criterio aggiudicazione non trovata")
            return
        
        pivot = df.pivot_table(
            index=Config.COL_YEAR, columns=Config.COL_AWARD, 
            values=Config.COL_PRICE, aggfunc='sum'
        ).fillna(0)
        
        fig, ax = plt.subplots(figsize=(14, 7))
        colors_criteria = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
        ax.stackplot(pivot.index, *pivot.T.values, 
                     labels=pivot.columns, alpha=0.75, colors=colors_criteria,
                     edgecolor='white', linewidth=1.5)
        
        ax.set_title("Evoluzione Criteri di Aggiudicazione (Valore Cumulativo)", 
                    fontweight='bold', fontsize=16, pad=15, color='#2C3E50')
        ax.set_xlabel("Anno", fontweight='bold', fontsize=12)
        ax.set_ylabel("Valore Totale (€)", fontweight='bold', fontsize=12)
        self._format_currency(ax, 'y', 'M')
        ax.legend(loc='upper left', framealpha=0.95, fontsize=10)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.show()
        self._save(fig, '12_criteria_evolution_stacked.png')

# --- INIZIALIZZAZIONE ANALYZER ---
temp_analyzer = TemporalAnalyzer()

### 2.1 Grafico 1: Heatmap Stagionale Mese × Anno

**Scelta Tecnica: Heatmap (Visualizzazione basata su Calendario)**

Questa heatmap è una potente visualizzazione che mappa una griglia 2D (Mese × Anno) a una variabile quantitativa (N. Contratti) usando il canale del **Colore (Saturazione/Luminosità)**.

Questo approccio è una diretta applicazione della tecnica **"Cluster and Calendar based Visualization"** descritta da van Wijk & van Selow.
1.  **Vista Calendario:** La griglia Mese/Anno *è* la vista calendario.
2.  **Percezione Preattentiva:** Invece di leggere una tabella di numeri, il nostro cervello usa l'elaborazione preattentiva del colore (un **attributo preattentivo**) per identificare *istantaneamente*:
    * **Pattern Verticali (Stagionalità):** Le righe rosse (alta intensità) alla fine di ogni anno (Q4).
    * **Pattern Orizzontali (Trend):** Se un mese specifico sta diventando più "caldo" (rosso) nel corso degli anni.
    * **Anomalie:** Singole celle "fredde" (blu) in un periodo "caldo" (rosso) o viceversa.

**Interpretazione Pattern Stagionali:**
* **Picchi Q4 (Nov-Dic):** Evidente concentrazione di contratti a fine anno, probabilmente dovuta a cicli di budget e scadenze amministrative.
* **Valle Estiva (Lug-Ago):** Calo di attività, coincidente con il periodo feriale.

In [ ]:
# --- ESECUZIONE HEATMAP STAGIONALE ---
temp_analyzer.plot_seasonality_heatmap(df_master)

### 2.2 Grafico 2: Evoluzione Criteri di Aggiudicazione (Stacked Area)

**Scelta Tecnica: Stacked Area Chart (Analisi Temporale "Part-to-Whole")**

Per analizzare come la *composizione* del mercato (i criteri di aggiudicazione) cambia nel tempo, usiamo uno Stacked Area Chart.

**Design Space:**
* **Asse X (Substrato):** Tempo (variabile **Quantitativa**).
* **Asse Y (Canale):** Valore € (variabile **Quantitativa**), mappata alla **Lunghezza/Posizione** verticale.
* **Aree (Segni):** Le aree rappresentano il volume monetario di ciascun criterio.
* **Colori (Canale):** La **Tonalità (Hue)** è usata per distinguere le categorie **Nominali** (i diversi criteri).

Questa tecnica è superiore a un semplice grafico a linee quando si analizza la composizione *part-to-whole*, perché mostra due cose contemporaneamente:
1.  **Trend Totale:** L'altezza complessiva del grafico mostra la crescita/contrazione del valore totale del mercato.
2.  **Composizione (Part-to-Whole):** La grandezza relativa di ogni area colorata mostra la quota di mercato di quel criterio in un dato anno.

**Interpretazione Shift Normativo:**
* **Crescita "Qualità/Prezzo"**: Si osserva un chiaro aumento della quota di questo criterio, probabilmente in linea con le direttive EU che spingono per l'offerta economicamente più vantaggiosa (MEAT) invece del mero "Prezzo Più Basso".
* **Riduzione "Prezzo Più Basso"**: Questo criterio, sebbene ancora presente, sta perdendo quota di mercato in termini di valore.

In [ ]:
# --- ESECUZIONE EVOLUZIONE CRITERI ---
temp_analyzer.plot_criteria_evolution(df_master)

## 3. Analisi Geospaziale (Zoom sulla Dimensione Territoriale)

Continuiamo la nostra fase di **"Zoom and Filter"**, spostando il focus dal *tempo* allo *spazio*. L'obiettivo è capire *dove* vengono allocati i fondi pubblici.

Utilizzeremo mappe coropletiche, che sono la tecnica di visualizzazione standard per **Dati 2D-Map** o **Dati Geografici**. Come evidenziato da esempi storici (la mappa del colera di John Snow, 1854), la visualizzazione spaziale è fondamentale per generare *insight* legati alla localizzazione.

In [ ]:
import json

class GeospatialAnalyzer(BasePlotter):
    """Analisi geografica."""

    def __init__(self):
        super().__init__()
        self.geojson = None
        self.district_mapping = {
            'Região Autónoma dos Açores': 'Açores',
            'Região Autónoma da Madeira': 'Madeira'
        }
        
        geojson_path = Config.GEOJSON.parent / 'georef-portugal-distrito.geojson'
        if geojson_path.exists():
            with open(geojson_path, 'r', encoding='utf-8') as f:
                self.geojson = json.load(f)

    def plot_choropleth(self, df: pd.DataFrame, metric_col: str, agg_func: str, title: str, filename: str):
        """Genera mappa coropletica."""
        if self.geojson is None: 
            return
        
        df_mapped = df.copy()
        if Config.COL_DISTRICT in df_mapped.columns:
            df_mapped[Config.COL_DISTRICT] = df_mapped[Config.COL_DISTRICT].replace(self.district_mapping)
        
        dist_data = df_mapped.groupby(Config.COL_DISTRICT)[metric_col].agg(agg_func).reset_index()
        dist_data.columns = [Config.COL_DISTRICT, 'Value']
        
        # Calcola centroidi per le etichette
        centroids = []
        for feat in self.geojson['features']:
            name = feat['properties']['dis_name']
            if name in dist_data[Config.COL_DISTRICT].values:
                geom = feat['geometry']['coordinates']
                if feat['geometry']['type'] == 'MultiPolygon':
                    coords = [pt for poly in geom for ring in poly for pt in ring]
                else:
                    coords = [pt for ring in geom for pt in ring]
                lons = [c[0] for c in coords]
                lats = [c[1] for c in coords]
                centroids.append({
                    'District': name,
                    'lon': sum(lons) / len(lons),
                    'lat': sum(lats) / len(lats)
                })
        
        fig = px.choropleth_mapbox(
            dist_data, geojson=self.geojson, locations=Config.COL_DISTRICT,
            featureidkey='properties.dis_name', color='Value',
            color_continuous_scale='Teal', mapbox_style="carto-positron",
            zoom=5.5, center={"lat": 39.5, "lon": -8.0}, opacity=0.75,
            hover_name=Config.COL_DISTRICT, hover_data={'Value': ':,.0f'}
        )
        
        # Aggiungi etichette distretto
        for c in centroids:
            fig.add_scattermapbox(
                lon=[c['lon']], lat=[c['lat']], mode='text',
                text=[c['District']], textfont=dict(size=10, color='black'),
                hoverinfo='skip', showlegend=False
            )
        
        fig.update_layout(title_text=title, margin={"r":0,"t":50,"l":0,"b":0}, height=600)
        
        html_path = Config.PLOTS_DIR / filename
        fig.write_html(html_path)
        fig.show()

    def plot_top_districts_bars(self, df: pd.DataFrame):
        """Top 15 distretti per Totale e Media."""
        metrics = df.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].agg(
            Total='sum', Average='mean'
        ).reset_index()

        top15_total = metrics.sort_values('Total', ascending=True).tail(15)
        top10_total = metrics.sort_values('Total', ascending=False).head(15)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [2, 1]})
        
        norm = plt.Normalize(vmin=top15_total['Total'].min(), vmax=top15_total['Total'].max())
        colors_total = plt.cm.Reds(norm(top15_total['Total']))
        ax1.barh(top15_total[Config.COL_DISTRICT], top15_total['Total']/1e6, color=colors_total, edgecolor='#2C3E50')
        ax1.set_title("Top 15 Distretti per Valore Totale", fontsize=14, weight='bold')
        ax1.set_xlabel("Valore Totale (€M)", weight='bold')
        
        sm = plt.cm.ScalarMappable(cmap='Reds', norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax1, label='Intensità Valore (€)').ax.yaxis.label.set_weight('bold')
        
        wedges, texts, autotexts = ax2.pie(
            top10_total['Total'], labels=top10_total[Config.COL_DISTRICT],
            autopct='%1.1f%%', startangle=90, counterclock=False,
            colors=plt.cm.Reds(np.linspace(0.8, 0.2, 15)),
            wedgeprops={'edgecolor': 'white', 'linewidth': 1}
        )
        ax2.set_title("Top 5 Distretti\n(Quota sul Totale dei Top 15)", weight='bold')
        plt.setp(autotexts, size=9, weight='bold', color='white')
        import matplotlib.patheffects as path_effects
        for text in autotexts:
            text.set_path_effects([path_effects.withStroke(linewidth=2, foreground='black')])

        plt.tight_layout()
        plt.show()
        self._save(fig, '15a_top15_districts_total.png')

        top15_avg = metrics.sort_values('Average', ascending=True).tail(15)
        top10_avg = metrics.sort_values('Average', ascending=False).head(15)
        
        fig, (ax3, ax4) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [2, 1]})
        
        norm_avg = plt.Normalize(vmin=top15_avg['Average'].min(), vmax=top15_avg['Average'].max())
        colors_avg = plt.cm.Blues(norm_avg(top15_avg['Average']))
        ax3.barh(top15_avg[Config.COL_DISTRICT], top15_avg['Average']/1e6, color=colors_avg, edgecolor='#2C3E50')
        ax3.set_title("Top 15 Distretti per Valore Medio", fontsize=14, weight='bold')
        ax3.set_xlabel("Valore Medio (€M)", weight='bold')
        
        sm_avg = plt.cm.ScalarMappable(cmap='Blues', norm=norm_avg)
        sm_avg.set_array([])
        plt.colorbar(sm_avg, ax=ax3, label='Intensità Valore (€)').ax.yaxis.label.set_weight('bold')
        
        wedges, texts, autotexts = ax4.pie(
            top10_avg['Average'], labels=top10_avg[Config.COL_DISTRICT],
            autopct='%1.1f%%', startangle=90, counterclock=False,
            colors=plt.cm.Blues(np.linspace(0.8, 0.2, 10)),
            wedgeprops={'edgecolor': 'white', 'linewidth': 1}
        )
        ax4.set_title("Top 15 Distretti\n(Confronto Medie)", weight='bold')
        plt.setp(autotexts, size=9, weight='bold', color='white')
        for text in autotexts:
            text.set_path_effects([path_effects.withStroke(linewidth=2, foreground='black')])

        plt.tight_layout()
        plt.show()
        self._save(fig, '15b_top15_districts_avg.png')

    def plot_scatter_volume_value(self, df: pd.DataFrame):
        """Genera scatter plot volume vs valore."""
        metrics = df.groupby(Config.COL_DISTRICT).agg(
            Total_Value=(Config.COL_PRICE, 'sum'),
            Average_Value=(Config.COL_PRICE, 'mean'),
            Contracts=(Config.COL_PRICE, 'count')
        ).reset_index()
        
        fig = px.scatter(
            metrics, x='Contracts', y='Total_Value', size='Average_Value',
            color=Config.COL_DISTRICT, hover_name=Config.COL_DISTRICT,
            size_max=50, title="Relazione Volume-Valore per Distretto",
            labels={'Contracts': 'Numero Contratti', 'Total_Value': 'Valore Totale (€)', 'Average_Value': 'Valore Medio (€)'},
            hover_data={'Contracts': True, 'Total_Value': ':,.0f', 'Average_Value': ':,.0f'}
        )
        fig.update_layout(height=700, plot_bgcolor='rgba(248,249,250,1)', font=dict(family="Arial", size=12))
        fig.write_html(Config.PLOTS_DIR / '16_scatter_districts_vol_val.html')
        fig.show()

geo_analyzer = GeospatialAnalyzer()

### 3.1 Grafico 1: Mappa Coropletica Valore Medio per Distretto

**Criteri di Scelta Tecnica (InfoVis su SciVis):**
* La mappa del Portogallo è il nostro **Substrato**, un dato con una mappatura geometrica intrinseca (ricade nella **SciVis**).
* Su questo substrato, noi mappiamo un dato **astratto** (il "Valore Medio" dei contratti), che non ha una posizione fisica. Questo processo di mappatura di dati astratti su una base geografica è un esempio classico di **InfoVis**.
* Il canale visivo utilizzato è il **Colore (Saturazione/Luminosità)**, dove un colore più intenso corrisponde a un valore medio più alto.

**Motivazione Metrica (Valore Medio vs. Totale):**
Iniziamo con il *Valore Medio* (invece del Totale) perché questo "normalizza" l'analisi. Ci dice la *tipologia* di contratto prevalente in un distretto (pochi grandi progetti vs. tanti piccoli progetti), piuttosto che mostrarci semplicemente dove c'è più popolazione (che è ciò che il Valore Totale solitamente mostra).

**Pattern Osservati:**
* Aree Metropolitane (Lisbona, Porto): Valori medi non necessariamente i più alti, indicando un mix di contratti grandi e piccoli.
* Distretti Costieri/Interni: Alcuni distretti con *basso volume* totale (come vedremo) mostrano un *alto valore medio*, suggerendo una concentrazione su pochi, grandi progetti infrastrutturali (es. dighe, autostrade).

In [ ]:
# --- ESECUZIONE: Mappa Valore Medio ---
geo_analyzer.plot_choropleth(
    df_master, 
    Config.COL_PRICE, 
    'mean', 
    "Valore Medio Contratti per Distretto (€)", 
    "13_map_mean_value.html"
)

### 3.2 Grafico 2: Mappa Coropletica Volume Contratti per Distretto

**Perché un Secondo Choropleth per il Volume:**
Visualizzare il volume (numero di contratti) separatamente dal valore è essenziale. Questo è un esempio di utilizzo di **viste multiple coordinate** (in questo caso, mentalmente coordinate) per esplorare dati multidimensionali.

**Concetto di Brushing and Linking:**
Confrontando questa mappa (Volume) con la precedente (Valore Medio), applichiamo un "brushing" mentale, una tecnica di interazione fondamentale (anche se qui manuale):
1.  **Insight dalla Mappa 1:** "Beja ha un valore medio alto".
2.  **Check sulla Mappa 2:** "Beja ha un volume di contratti bassissimo".
3.  **Insight Combinato (Cognizione):** L'alto valore medio di Beja è guidato da pochissimi progetti di grande entità, non da un'economia locale vivace. È un mercato "di nicchia".

**Interpretazione del Volume:**
* **Dominio di Lisbona**: Come previsto, la capitale domina per numero di contratti, sede delle autorità centrali.
* **Confronto (Insight Chiave):**
    * **Lisbona**: ALTO volume + MEDIO valore medio = Mercato maturo, frammentato, con molti servizi e manutenzioni.
    * **Distretti Interni (es. Beja, Évora)**: BASSO volume + ALTO valore medio = Mercato di nicchia, specializzato in grandi opere.

In [ ]:
# --- ESECUZIONE: Mappa Volume ---
geo_analyzer.plot_choropleth(
    df_master, 
    Config.COL_PRICE, 
    'count', 
    "Volume Contratti per Distretto (N. Contratti)", 
    "14_map_volume.html"
)

### 3.3 Grafico 3: Top 15 Distretti - Bar Chart

Dopo l'analisi geografica (che usa il *colore*), torniamo alla visualizzazione più efficace per il confronto quantitativo e il ranking: il **Bar Chart**.

**Criteri di Scelta Tecnica:**
* **Bar Chart Orizzontale**: Mappa il Valore Totale/Medio alla **Lunghezza** su un asse comune. È la scelta migliore per un ranking preciso. L'orientamento orizzontale è preferito per la leggibilità delle etichette **Nominali** (i nomi dei distretti).
* **Colorbar (Encoding Ridondante)**: Il colore (saturazione) è usato in modo **ridondante** per mappare lo stesso valore della lunghezza. Questo rafforza il pattern visivo. Come discusso da Borkin et al. ("Beyond Memorability"), la **ridondanza** (sia di dati che di messaggio) è correlata positivamente a una migliore qualità del richiamo.
* **Pie Chart (Top 5)**: Inclusa per un rapido colpo d'occhio sulla *composizione part-to-whole*. Tuttavia, è una tecnica percettivamente debole (usa **Angolo** e **Area**, canali imprecisi) e viene usata solo come supporto al bar chart principale.

**Analisi Valore Totale vs. Medio:**
* **Valore Totale:** Mostra una concentrazione estrema (es. Lisbona, Porto).
* **Valore Medio:** Il ranking si *inverte*. Distretti con basso volume totale (come visto nelle mappe) scalano la classifica del valore medio, confermando l'ipotesi dei "pochi grandi progetti".

In [ ]:
# --- ESECUZIONE: Bar Charts Top 15 ---
geo_analyzer.plot_top_districts_bars(df_master)

### 3.4 Grafico 4: Scatter Plot Multivariato Volume vs Valore

Questo scatter plot è una visualizzazione di **dati Multidimensionali (n-D)** che sintetizza 4 dimensioni in un unico grafico 2D, massimizzando la densità di informazione.

**Design Space (Mappatura dei Canali):**
1.  **Asse X (Posizione)**: Numero Contratti (Quantitativo)
2.  **Asse Y (Posizione)**: Valore Totale (€) (Quantitativo)
3.  **Dimensione Bolla (Area)**: Valore Medio (€) (Quantitativo)
4.  **Colore (Tonalità)**: Distretto (Nominale)

**Analisi (4 Archetipi Distrettuali):**
Questa visualizzazione è eccellente per identificare *cluster* e *outlier*.
1.  **Mercati Maturi (Alto X, Alto Y):** Lisbona, Porto. In alto a destra.
2.  **Opportunità Specialistiche (Basso X, Alto Y, Bolla Grande):** Pochi contratti, ma alto valore totale e medio.
3.  **Mercati Emergenti (Medio-Medio):** Centro del grafico.
4.  **Mercati Periferici (Basso X, Basso Y):** In basso a sinistra.

Questo grafico è la sintesi perfetta dell'analisi geospaziale, confermando gli insight ottenuti combinando le due mappe coropletiche.

In [ ]:
# --- ESECUZIONE: Scatter Plot Volume vs Valore ---
geo_analyzer.plot_scatter_volume_value(df_master)

## 4. Approfondimenti Finanziari (Zoom sulla Distribuzione)

In questa sezione, applichiamo uno **"Zoom"** sulla variabile più importante del dataset: il Prezzo (`Base Bid Price`). L'obiettivo è andare oltre le medie e capire la *forma* e la *struttura* delle distribuzioni finanziarie, evitando le trappole statistiche come quelle del **Quartetto di Anscombe**.

In [ ]:
class FinancialAnalyzer(BasePlotter):
    """Analisi delle distribuzioni finanziarie e relazioni prezzo-criteri."""

    def plot_price_distribution_log(self, df: pd.DataFrame):
        """Genera istogramma prezzi con scala logaritmica (Matplotlib)."""
        fig, ax = plt.subplots(figsize=(12, 7))
        
        # Istogramma con bins logaritmici
        prices = df[Config.COL_PRICE][df[Config.COL_PRICE] > 0]
        ax.hist(prices, bins=np.logspace(np.log10(prices.min()), np.log10(prices.max()), 80),
                color='#3498db', alpha=0.7, edgecolor='black', linewidth=0.5)
        
        ax.set_xscale('log')
        ax.set_title("Distribuzione Prezzo Base (Scala Logaritmica)", fontsize=14, weight='bold')
        ax.set_xlabel("Prezzo Base (€, scala log)", fontsize=12)
        ax.set_ylabel("Conteggio Contratti", fontsize=12)
        ax.grid(True, alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.show()
        self._save(fig, '17_price_distribution_log.png')

    def plot_price_by_criteria_box(self, df: pd.DataFrame):
        """Genera box plot confronto prezzi per criterio (Seaborn)."""
        if Config.COL_AWARD not in df.columns: 
            print("Colonna criterio aggiudicazione non trovata")
            return
        
        fig, ax = plt.subplots(figsize=(12, 7))
        
        # Box plot con palette professionale (blu/rosso/grigio)
        df_plot = df[df[Config.COL_PRICE] > 0].copy()
        colors = ['#3498DB', '#E74C3C', '#95A5A6'][:df_plot[Config.COL_AWARD].nunique()]
        
        sns.boxplot(data=df_plot, x=Config.COL_AWARD, y=Config.COL_PRICE, 
                    palette=colors, ax=ax, notch=True, linewidth=1.5)
        
        ax.set_yscale('log')
        ax.set_title("Confronto Prezzi per Criterio di Aggiudicazione (Scala Log)", 
                    fontsize=15, weight='bold', pad=15)
        ax.set_xlabel("Criterio Aggiudicazione", fontsize=12, weight='bold')
        ax.set_ylabel("Prezzo Base (€, scala log)", fontsize=12, weight='bold')
        ax.tick_params(axis='x', rotation=15)
        ax.grid(True, alpha=0.3, linestyle='--', axis='y')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        # Legenda con nomi criteri
        legend_labels = [f'Criterio {i}' for i in sorted(df_plot[Config.COL_AWARD].unique())]
        ax.legend(handles=[plt.Rectangle((0,0),1,1, color=c) for c in colors[:len(legend_labels)]], 
                 labels=legend_labels, loc='upper right', frameon=True, fontsize=10)
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '18_price_by_criteria_box.png')

    def plot_price_intensity(self, df: pd.DataFrame):
        """Genera joint plot KDE per analisi densità bivariata prezzo-intensità con legenda."""
        if Config.COL_PRICE_DAY not in df.columns: 
            print("Colonna Price per Day non trovata")
            return
            
        plot_data = df[(df[Config.COL_PRICE] > 1) & (df[Config.COL_PRICE_DAY] > 1)]
        
        g = sns.jointplot(
            data=plot_data, x=Config.COL_PRICE, y=Config.COL_PRICE_DAY,
            kind="kde", fill=True, cmap='Blues', height=10,
            log_scale=(True, True)
        )
        g.fig.suptitle(
            "Intensità Economica: Prezzo Totale vs Prezzo/Giorno", 
            y=1.02, fontweight='bold', fontsize=16
        )
        g.set_axis_labels(
            "Prezzo Totale (€, log)", 
            "Costo Giornaliero (€/giorno, log)",
            fontsize=14
        )
        
        # Aggiungi testo esplicativo come legenda
        g.ax_joint.text(
            0.05, 0.95, 
            'Zone scure = Alta densità contratti\nZone chiare = Bassa densità contratti',
            transform=g.ax_joint.transAxes,
            fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='#2C3E50', linewidth=1.5),
            weight='bold'
        )
        
        plt.show()
        self._save(g.fig, '19_price_intensity_kde.png')

    def plot_budget_treemap(self, df: pd.DataFrame):
        if Config.COL_AWARD not in df.columns: 
            print("Colonna criterio aggiudicazione non trovata")
            return

        df_tree = df.groupby([Config.COL_DISTRICT, Config.COL_AWARD])[Config.COL_PRICE].sum().reset_index()
        df_tree = df_tree[df_tree[Config.COL_PRICE] > 0]

        district_totals = df_tree.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].sum()
        df_tree['Pct_in_District'] = df_tree.apply(
            lambda row: row[Config.COL_PRICE] / district_totals[row[Config.COL_DISTRICT]] * 100, axis=1
        )

        df_tree['Label'] = df_tree.apply(
            lambda row: f"{row[Config.COL_DISTRICT]}<br>{row[Config.COL_AWARD]} ({row['Pct_in_District']:.1f}%)",
            axis=1
        )

        fig = px.treemap(
            df_tree,
            path=[px.Constant("Portogallo"), Config.COL_DISTRICT, Config.COL_AWARD],
            values='Pct_in_District',
            color='Pct_in_District',
            color_continuous_scale='RdBu',
            title="Allocazione Percentuale Budget: Distretto > Criterio Aggiudicazione",
            hover_data={
                'Pct_in_District': ':.1f'
            }
        )
        fig.update_traces(
            texttemplate="<b>%{label}</b><br>%{value:.1f}%",
            textposition="middle center"
        )
        fig.update_layout(
            height=800,
            font=dict(family="Arial", size=11)
        )
        fig.write_html(Config.PLOTS_DIR / '20_budget_treemap.html')
        fig.show()
  

# --- INIZIALIZZAZIONE ANALYZER ---
fin_analyzer = FinancialAnalyzer()

### 4.1 Grafico 1: Distribuzione Prezzo (Scala Logaritmica)

**Scelta Tecnica: Istogramma con Scala Logaritmica**

**Perché non una scala lineare?** Come identificato nella Fase 1, la distribuzione dei prezzi è estremamente *right-skewed* (asimmetrica a destra). Un istogramma in scala lineare comprimerebbe il 99% dei dati in poche barre a sinistra, rendendo impossibile l'analisi.

**Perché una scala logaritmica?**
La scala logaritmica è una *trasformazione* dei dati che "comprime" gli ordini di grandezza. Ci permette di vedere simultaneamente la distribuzione dei micro-contratti (es. €1k-€10k) e dei mega-progetti (es. €10M-€100M) nello stesso grafico.

**Interpretazione della Distribuzione (Bimodalità):**
La visualizzazione rivela una distribuzione che non è unimodale, ma sembra avere almeno due "picchi" (bimodale):
1.  **Moda Primaria (es. €50k-€200k):** Il "pane quotidiano" degli appalti (manutenzioni, piccoli lavori).
2.  **Coda Lunga (es. >€5M):** I grandi progetti infrastrutturali.

Questo insight (bimodalità) era *invisibile* usando le sole statistiche di riepilogo (media/mediana), ribadendo l'importanza di *visualizzare i dati*.

In [ ]:
# Generazione distribuzione prezzi con scala logaritmica
fin_analyzer.plot_price_distribution_log(df_master)

### 4.1b Grafico 1b: Distribuzione Prezzo per Categoria (Violin Plot)

**Scelta Tecnica: Violin Plot (Istogramma + Box Plot)**

Per confrontare le distribuzioni di prezzo tra diverse categorie (in questo caso, le categorie di *durata* che abbiamo creato), il **Violin Plot** è una scelta eccellente.

Combina i vantaggi:
1.  **Box Plot (interno):** Mostra i robusti indicatori statistici (mediana, quartili) mappati alla **Posizione**.
2.  **KDE (Kernel Density Estimation) (la "forma" del violino):** Mostra la *forma* della distribuzione (come un istogramma specchiato e smussato).

Questo ci permette di confrontare non solo le mediane, ma anche la *forma* delle distribuzioni per ogni categoria (es. "I progetti 'Long' sono bimodali?"). L'uso della scala logaritmica sull'asse Y è di nuovo fondamentale.

In [ ]:
# Analisi distribuzione prezzo per categoria (Seaborn violin plot)
price_cat_col = 'Base Bid Price (€)_category'
if price_cat_col in df_master.columns and Config.COL_PRICE in df_master.columns:
    df_plot = df_master[[price_cat_col, Config.COL_PRICE]].dropna()
    df_plot = df_plot[df_plot[Config.COL_PRICE] > 1]
    
    fig, ax = plt.subplots(figsize=(12, 7))
    # Palette moderna rosa-viola (no verdini)
    colors = ['#FCE4EC', '#F48FB1', '#EC407A', '#C2185B', '#880E4F'][:df_plot[price_cat_col].nunique()]
    sns.violinplot(data=df_plot, x=price_cat_col, y=Config.COL_PRICE, 
                   palette=colors, inner='box', ax=ax, linewidth=1.5)
    ax.set_yscale('log')
    ax.set_title("Distribuzione Prezzo per Durata Progetto", fontsize=15, weight='bold', pad=15)
    ax.set_xlabel("Categoria Durata", fontsize=12, weight='bold')
    ax.set_ylabel("Prezzo Base (€, scala log)", fontsize=12, weight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, linestyle='--', axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()
    fin_analyzer._save(fig, '17b_price_by_category_violin.png')
else:
    # Fallback: mostra distribuzione prezzo vs durata con bin categorie
    if Config.COL_DEADLINE in df_master.columns and Config.COL_PRICE in df_master.columns:
        df_plot = df_master[[Config.COL_DEADLINE, Config.COL_PRICE]].dropna()
        df_plot = df_plot[(df_plot[Config.COL_PRICE] > 1) & (df_plot[Config.COL_DEADLINE] > 1)]
        
        # Crea categorie durata
        df_plot['Duration_Category'] = pd.cut(
            df_plot[Config.COL_DEADLINE],
            bins=[0, 90, 180, 365, 730, float('inf')],
            labels=['<3 mesi', '3-6 mesi', '6-12 mesi', '1-2 anni', '>2 anni']
        )
        
        # Verifica distribuzione nelle categorie
        print("\nDistribuzione contratti per categoria durata:")
        print(df_plot['Duration_Category'].value_counts())
        print(f"\nStatistiche categoria '>2 anni':")
        cat_over_2y = df_plot[df_plot['Duration_Category'] == '>2 anni']
        if len(cat_over_2y) > 0:
            print(f"  Numero contratti: {len(cat_over_2y)}")
            print(f"  Prezzo min: €{cat_over_2y[Config.COL_PRICE].min():,.0f}")
            print(f"  Prezzo max: €{cat_over_2y[Config.COL_PRICE].max():,.0f}")
            print(f"  Prezzo mediano: €{cat_over_2y[Config.COL_PRICE].median():,.0f}")
        
        fig, ax = plt.subplots(figsize=(14, 8))
        # Palette blu sfumato (no verdini) con colori più intensi
        colors = ['#D1E5F0', '#92C5DE', '#4393C3', '#2166AC', '#053061']
        sns.violinplot(data=df_plot, x='Duration_Category', y=Config.COL_PRICE,
                       palette=colors, inner='box', ax=ax, linewidth=1.5, cut=0, scale='width')
        ax.set_yscale('log')
        ax.set_title("Distribuzione Prezzo per Durata Progetto", fontsize=16, weight='bold', pad=20)
        ax.set_xlabel("Categoria Durata", fontsize=13, weight='bold')
        ax.set_ylabel("Prezzo Base (€, scala log)", fontsize=13, weight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=11)
        ax.tick_params(axis='y', labelsize=11)
        ax.grid(True, alpha=0.3, linestyle='--', axis='y')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        # Aggiungi conteggi su ogni violin
        for i, cat in enumerate(['<3 mesi', '3-6 mesi', '6-12 mesi', '1-2 anni', '>2 anni']):
            count = len(df_plot[df_plot['Duration_Category'] == cat])
            ax.text(i, ax.get_ylim()[1]*0.9, f'n={count}', 
                   ha='center', fontsize=10, weight='bold', color='#2C3E50',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='gray'))
        plt.tight_layout()
        plt.show()
        fin_analyzer._save(fig, '17b_price_by_duration_violin.png')
    else:
        print("Colonne necessarie per analisi prezzo/durata non trovate")

### 4.2 Grafico 2: Box Plot Notched per Criteri di Aggiudicazione

**Scelta Tecnica: Notched Box Plot (Significatività Statistica)**

Questo grafico risponde alla domanda: "I contratti aggiudicati con criterio 'MEAT' (Qualità/Prezzo) valgono *davvero* di più di quelli 'Prezzo Più Basso'?"

Un **Notched Box Plot** è superiore a un box plot standard perché aggiunge un "test di ipotesi" visivo:
* **Box Plot:** Mappa le statistiche (mediana, quartili) alla **Posizione**.
* **Notches (Incavi):** Rappresentano l'intervallo di confidenza al 95% attorno alla mediana.

**Interpretazione Statistica:**
Se le *notches* di due box **non si sovrappongono**, c'è una forte evidenza statistica (p < 0.05) che le mediane dei due gruppi siano significativamente diverse.

**Interpretazione dei Dati:**
* Il grafico mostra che le *notches* tra "MEAT" e "Prezzo Più Basso" non si sovrappongono affatto.
* **Insight:** Possiamo affermare con confidenza statistica che, sì, i contratti basati sulla qualità hanno una mediana di valore significativamente superiore. Questo supporta l'ipotesi che i criteri qualitativi siano riservati ai progetti più grandi e complessi.

In [ ]:
# Confronto prezzi per criterio di aggiudicazione con notch per significatività
fin_analyzer.plot_price_by_criteria_box(df_master)

### 4.3 Grafico 3: Joint Plot KDE - Analisi Densità Bivariata

Questo è uno dei grafici più densi di informazione. Risponde alla domanda: "Esiste una relazione tra il Prezzo Totale di un contratto e il suo Costo Giornaliero?"

**Scelta Tecnica: Joint Plot (KDE) su Scala Log-Log**
* **Problema (Overplotting):** Con migliaia di punti, un semplice scatter plot sarebbe una nuvola nera illeggibile (overplotting).
* **Soluzione (KDE):** Un **Kernel Density Estimation (KDE)** plot stima la densità dei punti e la mappa al canale del **Colore (Saturazione)**. Le aree scure (alta densità) mostrano dove si concentrano i contratti, rivelando i *cluster*.
* **Joint Plot:** I grafici marginali (istogrammi/KDE sui lati) mostrano le distribuzioni 1D delle singole variabili, fornendo un contesto (Overview) all'analisi di correlazione centrale (Detail).
* **Scala Log-Log:** Necessaria per gestire la *skewness* su entrambi gli assi.

**Interpretazione (Archetipi di Intensità Economica):**
Il grafico rivela diversi cluster:
1.  **"Quick Wins" (Basso Totale, Alto Costo/Giorno):** In basso a destra. Interventi brevi e costosi (es. riparazioni urgenti).
2.  **"Progetti Maratona" (Alto Totale, Basso Costo/Giorno):** In alto a sinistra. Grandi infrastrutture diluite su molti anni.
3.  **"Standard Operativi" (Centro):** La massa principale dei contratti (es. medio valore, media durata).

In [ ]:
# Analisi densità bivariata: relazione tra prezzo totale e intensità economica giornaliera
fin_analyzer.plot_price_intensity(df_master)

### 4.4 Grafico 4: Treemap Gerarchico - Allocazione Budget

Questo grafico mostra la relazione *part-to-whole* del budget, ma su due livelli di gerarchia: Distretto -> Criterio.

**Scelta Tecnica: Treemap (Visualizzazione Gerarchica)**
* Un **Treemap** è la tecnica di visualizzazione principe per i **Dati Gerarchici (Tree)**, introdotta da Shneiderman.
* È una tecnica *space-filling* (riempimento dello spazio) che mappa una variabile **Quantitativa** (il Budget) al canale dell'**Area** dei rettangoli.
* L'algoritmo di layout (probabilmente *Squarified*) cerca di mantenere i rettangoli il più "quadrati" possibile per facilitare il confronto delle aree (che è comunque un canale percettivamente difficile, meno efficace della lunghezza).

**Interpretazione (Details on Demand):**
L'interattività di Plotly rende questo grafico un perfetto esempio di **"Details on Demand"**.
* **Overview:** A colpo d'occhio, vediamo che Lisbona e Porto (i rettangoli più grandi) dominano.
* **Zoom:** Cliccando su un distretto (es. Lisbona), il grafico "zooma" per mostrare solo la composizione interna di quel distretto.
* **Details:** L'hover fornisce i valori numerici esatti.

**Pattern Osservati:**
* La composizione interna (il mix di Criteri) varia significativamente tra distretti, suggerendo specializzazioni territoriali.

In [ ]:
# Visualizzazione gerarchica allocazione budget: Distretto > Criterio di Aggiudicazione
fin_analyzer.plot_budget_treemap(df_master)

### 4.5 Grafico 5: Distribuzione Criteri di Aggiudicazione (Bar + Pie)

**Scelta Tecnica: Visualizzazione Duale (Ridondanza Efficace)**

Questa visualizzazione combina due grafici per mostrare la stessa partizione (la distribuzione dei Criteri).

1.  **Bar Chart (Sinistra):** Come sempre, il bar chart (mappatura a **Lunghezza**) è il modo più accurato e onesto per mostrare i confronti quantitativi e i ranking. È la nostra "fonte di verità" analitica.
2.  **Pie Chart (Destra):** Il pie chart (mappatura ad **Angolo/Area**) è percettivamente inferiore. Tuttavia, è universalmente compreso per comunicare una singola idea: la *composizione part-to-whole*.

**Motivazione (Memorabilità e Comunicazione):**
Includiamo entrambi perché si rivolgono a scopi diversi: il Bar Chart per l'**Analisi** (precisione), il Pie Chart per la **Comunicazione** (impatto immediato). Lo studio "Beyond Memorability" suggerisce che la **ridondanza** (mostrare dati e messaggi in più modi) e l'uso di "ganci visivi" (come un pie chart familiare) possono migliorare la comprensione e il *recall*.

In [ ]:
# Distribuzione criteri aggiudicazione: Barplot (quantità) + Pie Chart (percentuali)
if Config.COL_AWARD in df_master.columns:
    award_dist = df_master[Config.COL_AWARD].value_counts()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [1.2, 1]})
    
    # Palette professionale
    colors = ['#2C3E50', '#E74C3C', '#3498DB'][:len(award_dist)]
    
    # --- BARPLOT QUANTITÀ ASSOLUTE ---
    bars = ax1.barh(award_dist.index, award_dist.values, color=colors, edgecolor='white', linewidth=2)
    ax1.set_xlabel("Numero Contratti", fontsize=12, weight='bold')
    ax1.set_ylabel("Criterio Aggiudicazione", fontsize=12, weight='bold')
    ax1.set_title("Quantità Assolute per Criterio", fontsize=14, weight='bold', pad=15)
    ax1.grid(True, alpha=0.3, linestyle='--', axis='x')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    # Aggiungi valori sulle barre
    for i, (idx, val) in enumerate(award_dist.items()):
        ax1.text(val + max(award_dist.values)*0.02, i, f'{int(val):,}', 
                va='center', fontsize=11, weight='bold', color='#2C3E50')
    
    # --- PIE CHART PERCENTUALI ---
    import matplotlib.patheffects as path_effects
    wedges, texts, autotexts = ax2.pie(
        award_dist, 
        labels=award_dist.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors,
        wedgeprops=dict(edgecolor='white', linewidth=2.5),
        textprops={'fontsize': 11, 'weight': 'bold'}
    )
    
    # Testo percentuali con contorno nero per leggibilità
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(12)
        autotext.set_weight('bold')
        autotext.set_path_effects([
            path_effects.withStroke(linewidth=2, foreground='black')
        ])
    
    ax2.set_title("Distribuzione Percentuale", fontsize=14, weight='bold', pad=15)
    plt.tight_layout()
    plt.show()
    fin_analyzer._save(fig, '20b_award_criteria_pie.png')
else:
    print("Colonna criteri aggiudicazione non trovata")

## 5. Analisi Testuale e Semantica (Zoom sul "Cosa")

Dopo aver analizzato *Quanto*, *Quando* e *Dove*, ora "zoomiamo" sul *Cosa*: qual è l'oggetto dei contratti? In questa sezione, visualizziamo i risultati della nostra pipeline NLP (Fase 1, Sezione 7).

L'obiettivo è visualizzare i **Dati 1D-Lineari** (testo) che abbiamo trasformato in *feature* strutturate.

In [ ]:
from wordcloud import WordCloud

class TextAnalyzer(BasePlotter):
    """Analisi NLP per estrazione temi, clustering semantico e visualizzazione keyword."""
    
    def plot_keyword_stats(self, df: pd.DataFrame):
        """Genera bar chart + pie chart per top keyword (Matplotlib/Seaborn)."""
        kw_cols = [c for c in df.columns if c.startswith('cpvs_keyword_')]
        if not kw_cols: 
            print("Colonne keyword non trovate")
            return

        kw_counts = df[kw_cols].sum().sort_values(ascending=True)
        kw_counts.index = kw_counts.index.str.replace('cpvs_keyword_', '').str.replace('_', ' ')

        # Bar Chart orizzontale con gradient professionale
        fig1, ax1 = plt.subplots(figsize=(12, max(8, len(kw_counts)*0.3)))
        # Gradient da blu scuro a blu chiaro (no verdini)
        colors = plt.cm.Blues(np.linspace(0.5, 0.95, len(kw_counts)))
        bars = ax1.barh(kw_counts.index, kw_counts.values, color=colors, edgecolor='#2C3E50', linewidth=0.8)
        
        # Valori alla fine delle barre
        for i, (idx, val) in enumerate(kw_counts.items()):
            ax1.text(val + max(kw_counts.values)*0.01, i, f'{int(val)}', 
                    va='center', fontsize=10, weight='bold', color='#2C3E50')
        
        ax1.set_title("Frequenza Top Keyword nei Contratti (TF-IDF)", fontsize=15, weight='bold', pad=15)
        ax1.set_xlabel("Occorrenze Aggregate", fontsize=12, weight='bold')
        ax1.set_ylabel("Termini Estratti", fontsize=12, weight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--', axis='x')
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
        self._save(fig1, '21a_keyword_frequency_bar.png')

        # Donut Chart con palette moderna
        fig2, ax2 = plt.subplots(figsize=(11, 8))
        # Palette categorica moderna (no verdini)
        colors_pie = ['#2C3E50', '#E74C3C', '#3498DB', '#F39C12', '#9B59B6', 
                     '#1ABC9C', '#34495E', '#E67E22', '#95A5A6', '#16A085'][:len(kw_counts)]
        
        wedges, texts, autotexts = ax2.pie(
            kw_counts.values, 
            labels=kw_counts.index,
            autopct='%1.1f%%',
            startangle=90,
            colors=colors_pie,
            wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2.5),
            textprops={'fontsize': 10, 'weight': 'bold'}
        )
        
        # Percentuali con contorno per leggibilità
        import matplotlib.patheffects as path_effects
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontsize(11)
            autotext.set_weight('bold')
            autotext.set_path_effects([
                path_effects.withStroke(linewidth=2, foreground='black')
            ])
        
        # Legenda a destra
        ax2.legend(kw_counts.index, loc='center left', bbox_to_anchor=(1, 0.5), 
                  fontsize=10, frameon=True, title='Keywords', title_fontsize=11)
        
        ax2.set_title("Distribuzione Percentuale Keyword", fontsize=15, weight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        self._save(fig2, '21b_keyword_distribution_pie.png')

    def plot_wordcloud(self, df: pd.DataFrame, text_col: str):
        """Genera word cloud da testi preprocessati (lemmatized, stop words removed)."""
        if text_col not in df.columns: 
            print(f"Colonna {text_col} non trovata")
            return
            
        text = " ".join(df[text_col].dropna().astype(str))
        
        if len(text.strip()) < 100:
            print("Testo insufficiente per word cloud")
            return
            
        wc = WordCloud(
            width=1800, height=900, 
            background_color='black',  # Fondo nero per contrasto
            colormap='rainbow',  # Colori multipli ben leggibili
            max_words=200,
            relative_scaling=0.5,
            min_font_size=10,
            prefer_horizontal=0.7,
            contour_width=2,
            contour_color='white'  # Contorno bianco su fondo nero
        ).generate(text)
        
        fig, ax = plt.subplots(figsize=(18, 9))
        fig.patch.set_facecolor('black')  # Fondo figura nero
        ax.set_facecolor('black')  # Fondo axes nero
        ax.imshow(wc, interpolation='bilinear')
        ax.axis("off")
        ax.set_title(f"Word Cloud: {text_col}", fontsize=20, fontweight='bold', 
                    pad=20, color='white')  # Titolo bianco su fondo nero
        plt.tight_layout()
        plt.show()
        self._save(fig, '22_wordcloud.png')

    def plot_semantic_clusters(self, df: pd.DataFrame):
        """Scatter plot 2D dei cluster semantici da embeddings + PCA + K-Means."""
        req = ['semantic_x', 'semantic_y', 'semantic_cluster']
        if not all(c in df.columns for c in req): 
            print("Colonne semantic_x/y/cluster non trovate")
            return
            
        df_plot = df.dropna(subset=req).copy()
        df_plot['semantic_cluster'] = df_plot['semantic_cluster'].astype(str)
        
        fig = px.scatter(
            df_plot, x='semantic_x', y='semantic_y', 
            color='semantic_cluster',
            hover_data=[Config.COL_CPVS, Config.COL_PRICE] if Config.COL_CPVS in df.columns else None,
            title="Mappa Semantica Contratti: Cluster Tematici (PCA + K-Means)",
            labels={
                'semantic_x': 'Componente Principale 1',
                'semantic_y': 'Componente Principale 2',
                'semantic_cluster': 'Cluster'
            },
            color_discrete_sequence=px.colors.qualitative.Bold, 
            opacity=0.7
        )
        fig.update_layout(
            plot_bgcolor='rgba(248,249,250,1)',
            height=700,
            font=dict(family="Arial", size=12),
            legend_title_text='Cluster Semantico'
        )
        fig.update_traces(marker=dict(size=8, line=dict(width=0.5, color='white')))
        fig.write_html(Config.PLOTS_DIR / '23_semantic_clusters.html')
        fig.show()

# --- INIZIALIZZAZIONE ANALYZER ---
text_analyzer = TextAnalyzer()

### 5.1 Grafico 1: Frequenza Keyword da TF-IDF (Bar + Donut)

Questo grafico visualizza il *risultato* del nostro modello TF-IDF ("Analyze First").

**Scelta Tecnica (Bar + Donut):**
* **Bar Chart Orizzontale:** Come nel Grafico 4.5, il bar chart è lo strumento analiticamente corretto. Usa la **Lunghezza** per classificare in modo preciso l'importanza (punteggio TF-IDF aggregato) di ogni *keyword* estratta.
* **Donut Chart:** Variante del Pie Chart (sempre basata su **Angolo**), usata per comunicare l'idea di *composizione percentuale*.

**Interpretazione (Pattern Linguistici):**
Il grafico a barre ci dice quali sono i temi più *distintivi* e rilevanti.
* **Termini Infrastrutturali Core:** (es. "pavimentazione", "ristrutturazione"). Questi rappresentano il mercato *mainstream*.
* **Specializzazioni Tecniche:** (es. "impianti elettrici", "HVAC"). Questi rappresentano *nicchie* di mercato.

**Collegamento Teorico (VisRA):**
Questo approccio è concettualmente simile a **VisRA (Visual Readability Analysis)**. VisRA estrae feature linguistiche (complessità, lunghezza frase) e le visualizza per aiutare gli autori a revisionare i testi. Noi abbiamo estratto feature tematiche (TF-IDF) e le visualizziamo per aiutare gli analisti a "revisionare" la loro comprensione del mercato.

In [ ]:
# Analisi quantitativa frequenza keyword tramite TF-IDF
text_analyzer.plot_keyword_stats(df_master)

### 5.1b Grafico 1b: Keyword per Criterio di Aggiudicazione

**Scelta Tecnica: Grouped Bar Chart (Analisi Cross-Dimensionale)**

Questa è un'analisi più avanzata. Stiamo confrontando tre dimensioni:
1.  Keyword (Nominale)
2.  Criterio (Nominale)
3.  Occorrenze (Quantitativa)

Un **Grouped Bar Chart** è una tecnica efficace. L'asse X raggruppa per Keyword (la nostra variabile primaria di interesse), e il **Colore (Hue)** è usato per separare i Criteri all'interno di ogni gruppo. La **Lunghezza** della barra codifica le occorrenze.

**Interpretazione (Insight):**
Questo grafico può rivelare pattern strategici. Ad esempio:
* Le keyword "sostenibilità", "efficienza energetica" sono iper-rappresentate (barre alte) solo nei contratti "MEAT" (Qualità/Prezzo).
* Le keyword "manutenzione", "asfalto" dominano (barre alte) nei contratti "Prezzo Più Basso".
Questo fornisce una "mappa" per gli operatori su quale linguaggio usare nei *bid* a seconda del criterio di aggiudicazione.

In [ ]:
# Analisi keyword per criterio aggiudicazione (Matplotlib grouped bars)
if Config.COL_AWARD in df_master.columns:
    kw_cols = [c for c in df_master.columns if c.startswith('cpvs_keyword_')]
    
    if kw_cols:
        # Prendi top 10 keyword
        top_kw = df_master[kw_cols].sum().nlargest(10).index.tolist()
        
        # Crea matrice keyword × criterio
        cross_data = []
        for kw_col in top_kw:
            kw_name = kw_col.replace('cpvs_keyword_', '').replace('_', ' ')
            for criterion in df_master[Config.COL_AWARD].dropna().unique():
                mask = (df_master[Config.COL_AWARD] == criterion) & (df_master[kw_col] == 1)
                count = mask.sum()
                if count > 0:
                    cross_data.append({
                        'Keyword': kw_name,
                        'Criterio': criterion,
                        'Occorrenze': count
                    })
        
        if cross_data:
            df_cross = pd.DataFrame(cross_data)
            
            # Pivot per grouped bar chart
            pivot = df_cross.pivot(index='Keyword', columns='Criterio', values='Occorrenze').fillna(0)
            
            fig, ax = plt.subplots(figsize=(14, 8))
            # Palette professionale blu-rosso-grigio (no verdini)
            colors_criteria = ['#3498DB', '#E74C3C', '#95A5A6'][:len(pivot.columns)]
            pivot.plot(kind='bar', ax=ax, width=0.8, color=colors_criteria, edgecolor='#2C3E50', linewidth=0.8)
            
            ax.set_title("Top 10 Keyword per Criterio di Aggiudicazione", fontsize=15, weight='bold', pad=15)
            ax.set_xlabel("Keyword", fontsize=12, weight='bold')
            ax.set_ylabel("Occorrenze", fontsize=12, weight='bold')
            
            # Legenda a destra con etichette criteri
            legend_labels = [f'Criterio {col}' for col in pivot.columns]
            ax.legend(legend_labels, title='Criterio Aggiudicazione', 
                     loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, 
                     frameon=True, borderpad=1)
            
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, alpha=0.3, linestyle='--', axis='y')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            plt.show()
            text_analyzer._save(fig, '21c_keyword_by_criteria.png')
        else:
            print("Nessun dato per cross-analysis keyword-criteri")
    else:
        print("Colonne keyword non trovate")
else:
    print("Colonna criteri aggiudicazione non trovata")

### 5.2 Grafico 2: Word Cloud - Overview Visuale Temi Dominanti

**Criteri di Scelta Tecnica (e sue Limitazioni):**
* Una **Word Cloud** è una visualizzazione di **dati 1D-Lineari** (testo) molto popolare.
* Mappa la frequenza di una parola (Quantitativa) al canale visivo della **Dimensione del Font** (che è percepito come **Area**).
* **Limitazioni:** È una tecnica *percettivamente debole*. L'Area è un canale impreciso (in fondo alla gerarchia dei canali visivi), e la posizione delle parole è spesso casuale (non codifica nulla). È impossibile fare un confronto preciso.

**Perché usarla? (Memorabilità):**
La includiamo non per la sua precisione analitica, ma per il suo *impatto comunicativo*. È un eccellente "gancio visivo" (*visual hook*). Lo studio **"Beyond Memorability"** ha rilevato che visualizzazioni uniche e l'uso di "oggetti riconoscibili" (in questo caso, parole leggibili) possono aumentare la memorabilità e aiutare a comunicare il messaggio principale a colpo d'occhio, anche se in modo impreciso.

In [ ]:
# Generazione word cloud da testi preprocessati (lemmatized, stop words removed)
txt_col = 'Cpvs Designation_cleaned' if 'Cpvs Designation_cleaned' in df_master.columns else Config.COL_CPVS
text_analyzer.plot_wordcloud(df_master, txt_col)

### 5.3 Grafico 3: Clustering Semantico (Mappa Tematica)

Questo è il grafico finale della nostra pipeline NLP **"Analyze First"**. Visualizza i risultati del clustering semantico (Fase 1, Sezione 9).

**Pipeline e Design Space:**
1.  **Analyze First:** Abbiamo preso dati **1D-Lineari** (testo), li abbiamo trasformati in vettori **n-D** (Embeddings), e abbiamo calcolato i *cluster* (K-Means).
2.  **Visual Mapping:** Per visualizzare questi cluster, abbiamo ridotto le N dimensioni a 2 (usando PCA). Questo crea un **Substrato 2D** che *non è geografico*, ma *semantico*. Questo è un esempio puro di **InfoVis**: la posizione (lo *spatial layout*) è stata *creata* da noi per rivelare la struttura dei dati.
3.  **Mappatura Canali:**
    * Componente Principale 1 -> **Posizione Asse X**
    * Componente Principale 2 -> **Posizione Asse Y**
    * Cluster ID (Categoria) -> **Colore (Tonalità)**

**Interpretazione (Mappa Semantica):**
* Ogni punto è un contratto.
* I colori raggruppano i temi (es. "Infrastrutture Stradali", "Edifici Pubblici").
* La *distanza* tra i punti ha un significato: punti vicini sono semanticamente simili.
* Questo grafico ci permette di vedere la "geografia" del mercato degli appalti, non basata sul territorio, ma sui *temi*.

In [ ]:
# Visualizzazione mappa semantica: Sentence Embeddings → PCA 2D → K-Means Clustering
text_analyzer.plot_semantic_clusters(df_master)

## 6. Analisi Comparative e "Details on Demand"

Nell'ultima sezione, completiamo il mantra di Shneiderman fornendo visualizzazioni interattive e tabelle che abilitano l'esplorazione **"Details on Demand"**.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

class ComparativeAnalyzer(BasePlotter):
    """Analisi comparative avanzate con scatter plots interattivi e tabelle metriche."""
    
    def plot_price_intensity_interactive(self, df: pd.DataFrame):
        """Scatter plot interattivo Prezzo vs Costo/Giorno con marginals."""
        if Config.COL_PRICE_DAY not in df.columns or Config.COL_PRICE not in df.columns:
            print("Colonne necessarie non trovate")
            return None
        
        df_valid = df[(df[Config.COL_PRICE] > 1) & (df[Config.COL_PRICE_DAY] > 1)].copy()
        
        if len(df_valid) > 10000:
            df_valid = df_valid.sample(n=10000, random_state=42)
        
        fig = px.scatter(
            df_valid, 
            x=Config.COL_PRICE, 
            y=Config.COL_PRICE_DAY,
            color=Config.COL_AWARD if Config.COL_AWARD in df.columns else None,
            marginal_x='histogram',
            marginal_y='histogram',
            log_x=True, 
            log_y=True,
            opacity=0.65,
            title="Intensità Economica: Prezzo Totale vs Costo Giornaliero",
            hover_data=[Config.COL_DISTRICT, Config.COL_DEADLINE] if Config.COL_DISTRICT in df.columns else None,
            labels={
                Config.COL_PRICE: 'Prezzo Totale (€, log)',
                Config.COL_PRICE_DAY: 'Costo Giornaliero (€/giorno, log)',
                Config.COL_AWARD: 'Criterio'
            },
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        
        fig.update_layout(
            height=700, 
            showlegend=True,
            plot_bgcolor='rgba(248,249,250,1)',
            font=dict(family="Arial, sans-serif", size=12, color='#2C3E50')
        )
        fig.write_html(Config.PLOTS_DIR / '24_price_intensity_marginals.html')
        fig.show()
        
        # Ritorna statistiche per interpretazione esterna
        low_intensity = df_valid[df_valid[Config.COL_PRICE_DAY] < 1000]
        high_intensity = df_valid[df_valid[Config.COL_PRICE_DAY] > 10000]
        
        return {
            'total': len(df_valid),
            'low_intensity': len(low_intensity),
            'high_intensity': len(high_intensity),
            'low_pct': len(low_intensity)/len(df_valid)*100,
            'high_pct': len(high_intensity)/len(df_valid)*100
        }
    
    def generate_financial_metrics_table(self, df: pd.DataFrame):
        """Tabella metriche finanziarie aggregate per distretto (Pandas display)."""
        if Config.COL_DISTRICT not in df.columns or Config.COL_PRICE not in df.columns:
            print("Colonne necessarie non trovate")
            return None
        
        metrics = df.groupby(Config.COL_DISTRICT)[Config.COL_PRICE].agg([
            ('Valore_Totale', 'sum'),
            ('Valore_Medio', 'mean'),
            ('Valore_Mediano', 'median'),
            ('Num_Contratti', 'count'),
            ('Std_Dev', 'std')
        ]).reset_index()
        
        total_value = metrics['Valore_Totale'].sum()
        metrics['Quota_%'] = (metrics['Valore_Totale'] / total_value * 100).round(1)
        
        metrics = metrics.sort_values('Valore_Totale', ascending=False).head(15)
        
        # Formatta per display (converti in milioni)
        metrics_display = metrics.copy()
        metrics_display['Valore_Totale'] = (metrics_display['Valore_Totale']/1e6).round(1)
        metrics_display['Valore_Medio'] = (metrics_display['Valore_Medio']/1e6).round(2)
        metrics_display['Valore_Mediano'] = (metrics_display['Valore_Mediano']/1e6).round(2)
        metrics_display['Std_Dev'] = (metrics_display['Std_Dev']/1e6).round(2)
        
        # Rinomina colonne per display
        metrics_display.columns = ['Distretto', 'Valore Tot (€M)', 'Val Medio (€M)', 
                                   'Val Mediano (€M)', 'N. Contratti', 'Std Dev (€M)', 'Quota %']
        
        # Salva CSV
        metrics.to_csv(Config.PLOTS_DIR / 'metrics_by_district.csv', index=False)
        
        # Display con styling (compatibile nbconvert)
        print("\n=== Top 15 Distretti: Metriche Finanziarie Aggregate ===\n")
        display(metrics_display.style
                .background_gradient(subset=['Valore Tot (€M)'], cmap='Reds')
                .background_gradient(subset=['Quota %'], cmap='Blues')
                .format({'Valore Tot (€M)': '{:.1f}', 'Val Medio (€M)': '{:.2f}', 
                        'Val Mediano (€M)': '{:.2f}', 'Std Dev (€M)': '{:.2f}', 
                        'Quota %': '{:.1f}%'})
        )
        
        return metrics
    
    def plot_annual_volume_value_trend(self, df: pd.DataFrame):
        """Trend annuale dual-axis: volume (line) + valore (area) con Matplotlib."""
        if Config.COL_YEAR not in df.columns or Config.COL_PRICE not in df.columns:
            print("Colonne necessarie non trovate")
            return None
        
        yearly = df.groupby(Config.COL_YEAR).agg(
            Count=(Config.COL_YEAR, 'size'),
            Value=(Config.COL_PRICE, 'sum')
        ).reset_index()
        
        # Dual-axis matplotlib con colori migliorati
        fig, ax1 = plt.subplots(figsize=(14, 8))
        
        # Asse sinistro: Valore (area chart) - colore più tenue
        color1 = '#D73027'  # Rosso intenso ma non acceso
        ax1.set_xlabel('Anno', fontsize=13, weight='bold')
        ax1.set_ylabel('Valore Totale (€M)', fontsize=13, weight='bold', color=color1)
        ax1.fill_between(yearly[Config.COL_YEAR], yearly['Value']/1e6, 
                        alpha=0.25, color=color1, label='Valore Totale')
        ax1.plot(yearly[Config.COL_YEAR], yearly['Value']/1e6, 
                color=color1, linewidth=3, marker='o', markersize=9, 
                markerfacecolor='white', markeredgewidth=2.5, markeredgecolor=color1)
        ax1.tick_params(axis='y', labelcolor=color1, labelsize=11)
        ax1.grid(True, alpha=0.25, linestyle='--', color='gray')
        ax1.spines['top'].set_visible(False)
        
        # Asse destro: Volume (line chart) - blu più professionale
        ax2 = ax1.twinx()
        color2 = '#4575B4'  # Blu professionale
        ax2.set_ylabel('Numero Contratti', fontsize=13, weight='bold', color=color2)
        ax2.plot(yearly[Config.COL_YEAR], yearly['Count'], 
                color=color2, linewidth=3.5, marker='s', markersize=11, 
                markerfacecolor='white', markeredgewidth=2.5, markeredgecolor=color2,
                label='N. Contratti', linestyle='-')
        ax2.tick_params(axis='y', labelcolor=color2, labelsize=11)
        ax2.spines['top'].set_visible(False)
        
        # Titolo e legenda
        fig.suptitle("Trend Annuale: Volume vs Valore (Dual-Axis)", 
                    fontsize=16, weight='bold', color='#2C3E50')
        
        # Combina legende con frame
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', 
                  fontsize=12, frameon=True, fancybox=True, shadow=True)
        
        plt.tight_layout()
        plt.show()
        self._save(fig, '26_annual_volume_value_trend.png')
        
        # Calcola correlazione
        from scipy.stats import pearsonr
        corr, p_value = pearsonr(yearly['Count'], yearly['Value'])
        
        return {
            'correlation': corr,
            'p_value': p_value,
            'years': len(yearly)
        }

# --- INIZIALIZZAZIONE ANALYZER ---
comparative_analyzer = ComparativeAnalyzer()

### 6.1 Grafico 1: Scatter Plot Intensità Economica (Interattivo)

**Scelta Tecnica: Scatter Plot Interattivo con Marginal Plots**

Questa è una versione *interattiva* del grafico di densità KDE (Grafico 4.3).
* **Scatter Centrale:** Ci permette di selezionare singoli punti (contratti) e, tramite *hover* (il task **Details-on-demand**), vedere i loro dettagli specifici (es. distretto, durata).
* **Marginal Plots (Istogrammi):** Forniscono l'**Overview** delle distribuzioni 1D, dando contesto allo scatter plot 2D (i **Details**).
* **Interattività (Zoom/Filter):** La natura interattiva di Plotly permette all'utente di eseguire **Zoom** (geometrico) su un'area di interesse e **Filtrare** (cliccando sulla legenda) per isolare specifici criteri di aggiudicazione.

Questo grafico da solo implementa l'intero mantra di Shneiderman in un'unica interfaccia.

**Interpretazione Dati:**
* La colorazione per *Criterio* conferma visivamente che i contratti "Qualità/Prezzo" (MEAT) tendono a posizionarsi nella parte alta del grafico (valori totali e giornalieri più elevati).

In [ ]:
# --- ESECUZIONE SCATTER INTENSITÀ ---
stats_intensity = comparative_analyzer.plot_price_intensity_interactive(df_master)

if stats_intensity:
    print(f"Analisi Intensità Economica:")
    print(f"   • Contratti analizzati: {stats_intensity['total']:,}")
    print(f"   • Bassa intensità (<€1K/giorno): {stats_intensity['low_intensity']:,} ({stats_intensity['low_pct']:.1f}%)")
    print(f"   • Alta intensità (>€10K/giorno): {stats_intensity['high_intensity']:,} ({stats_intensity['high_pct']:.1f}%)")

### 6.2 Grafico 2: Tabella Metriche Finanziarie per Distretto

**Perché una Tabella in una Presentazione Visuale?**

Una tabella è la forma ultima di **"Details on Demand"**.
* **Limiti dei Grafici:** I grafici (come le mappe o i bar chart) sono eccellenti per mostrare *pattern*, *trend* e *confronti relativi* (fase di **Percezione**).
* **Forza delle Tabelle:** Sono ottimizzate per il *lookup* di valori esatti e la **Cognizione**. Se il mio compito è "Trovare il valore mediano esatto per il distretto di Braga", una tabella è infinitamente più veloce ed efficace di un grafico.

**Ridondanza Efficace (Borkin et al.):**
Lo studio "Beyond Memorability" ha evidenziato che il **testo è fondamentale** e che la **ridondanza dei dati** (Data Redundancy), come mostrare un valore in un grafico *e* in una tabella, migliora la comprensione e il *recall*.

Questa tabella, con *gradient* di colore (che è una mini-heatmap), funge da sommario quantitativo finale, fornendo i numeri esatti che stanno dietro ai pattern visuali che abbiamo osservato.

In [ ]:
# --- ESECUZIONE TABELLA METRICHE ---
metrics_df = comparative_analyzer.generate_financial_metrics_table(df_master)

if metrics_df is not None:
    top3_share = metrics_df.head(3)['Quota_%'].sum()
    print(f"\nConcentrazione Mercato:")
    print(f"   • Top 3 distretti = {top3_share:.1f}% valore totale")
    print(f"   • Livello concentrazione: {'ALTA (oligopolio)' if top3_share > 50 else 'MEDIA' if top3_share > 35 else 'BASSA (frammentato)'}")

### 6.3 Grafico 3: Trend Annuale Volume vs Valore (Dual-Axis)

**Cosa Visualizza:**
Riproponiamo il grafico dual-axis della dashboard (Grafico 1.1), ma come grafico statico (Matplotlib) finale. Questo grafico è l'epitome dell'analisi di **Dati Temporali (Time-Series)**.

**Motivazioni Scelte Tecniche:**
* **Dual-Axis:** Necessario per confrontare due variabili **Quantitative** con scale incomparabili (N. contratti vs. € Milioni) sullo stesso **Substrato** temporale (l'asse X).
* **Area (Valore) vs. Linea (Volume):** L'uso di *Segni* diversi (un'**Area** per il Valore e una **Linea** per il Volume) aiuta a distinguerli visivamente.
    * L'**Area** (rossa) dà un senso di "massa economica".
    * La **Linea** (blu) traccia il trend del volume.

**Interpretazione Dati (Correlazione):**
* **Correlazione Positiva (r=...):** L'analisi statistica (stampata sotto il grafico) conferma ciò che la visualizzazione mostra: una correlazione positiva e statisticamente significativa.
* **Analisi degli Sfasamenti (Time-Series):** Questo grafico sarebbe la base per un'analisi più approfondita, come il **Dynamic Time Warping (DTW)** (una misura di distanza robusta per le time-series), per vedere se, ad esempio, i picchi di *Volume* anticipano (o seguono) i picchi di *Valore*, o se ci sono *offset*.
* **Anomalie Temporali:** I picchi e le valli (es. 2016, 2020) sono chiaramente visibili e possono essere correlati a eventi esterni (cicli elettorali, fondi EU, pandemia), fornendo la base per ulteriori indagini (il ciclo **"Analyze Further"**).

In [ ]:
# --- ESECUZIONE TREND ANNUALE ---
trend_stats = comparative_analyzer.plot_annual_volume_value_trend(df_master)

if trend_stats:
    print(f"\nAnalisi Correlazione Volume-Valore:")
    print(f"   • Coefficiente correlazione Pearson: r = {trend_stats['correlation']:.3f}")
    print(f"   • Significatività statistica: p-value = {trend_stats['p_value']:.4f}")
    print(f"   • Anni analizzati: {trend_stats['years']}")
    
    if trend_stats['p_value'] < 0.05:
        strength = 'FORTE' if abs(trend_stats['correlation']) > 0.7 else 'MODERATA' if abs(trend_stats['correlation']) > 0.4 else 'DEBOLE'
        print(f"   • Interpretazione: Correlazione {strength} e statisticamente significativa")
        print(f"   • Implicazione: Mercato {'elastico - crescita economica si traduce in più contratti E più valore' if trend_stats['correlation'] > 0 else 'anelastico - dinamiche volume/valore disaccoppiate'}")

## 7. Conclusioni e Insight Strategici

L'intero processo di analisi, dal caricamento alla visualizzazione, ha seguito una metodologia strutturata (Keim e Shneiderman) per trasformare dati grezzi in *insight* azionabili.

L'applicazione rigorosa dei principi di InfoVis non è stata un esercizio accademico, ma l'unico modo per scoprire pattern che le sole statistiche di riepilogo avrebbero nascosto (come ampiamente dimostrato dall'**Anscombe's Quartet**).

**I 3 Insight Chiave Emersi:**

1.  **Il Mercato Non È Uniforme, È Bimodale:** L'analisi distributiva (Grafico 4.1) ha rivelato che non esiste un "mercato unico" degli appalti. Esistono almeno due mercati distinti: un "mercato di massa" (alto volume, basso valore, es. manutenzioni) e un "mercato di nicchia" (basso volume, altissimo valore, es. grandi infrastrutture).

2.  **La Geografia è Specializzazione, Non Solo Volume:** Le sole mappe di "Volume" o "Valore Totale" sono ingannevoli (mostrano solo la popolazione). L'incrocio di **viste multiple** (Mappe + Scatter Plot 3.4) ha rivelato gli archetipi distrettuali: mercati "maturi" (Lisbona) vs. mercati "specialistici" (es. distretti interni con BASSO volume ma ALTO valore medio).

3.  **I "Temi" Guidano il Valore:** L'applicazione del **"Analyze First"** tramite NLP (Grafici 5.1-5.3) ha mappato le competenze richieste. L'incrocio con l'analisi finanziaria (Grafico 5.1b) ha confermato che i criteri di aggiudicazione (es. "MEAT") sono fortemente correlati a temi specifici (es. "sostenibilità"), permettendo di capire *cosa* il mercato è disposto a pagare di più.

In conclusione, questo caso di studio ha dimostrato come un'analisi visuale non serva a produrre "bei grafici", ma a costruire un **motore di generazione di insight** per supportare decisioni strategiche.

In [ ]:
!jupyter nbconvert --to html --no-input --template lab preProcessData_story_ref.ipynb